In [1]:
import pandas as pd
import re
import json


# ============================================================
# 1. CONFIGURATION: KEYWORDS USED FOR TRANSPARENT CLASSIFICATION
# ============================================================

HEADER_TERMS = {
    "components",
    "subcomponents",
    "themes",
    "parameters",
    "facets",
    "styles",
    "behaviors",
    "end points",
    "endpoints",
    "drivers",
}


MEDICAL_KEYWORDS = {
    "fsh",
    "hormone",
    "diagnosis",
    "sleep apnea",
    "metabolic",
    "immune",
    "basophil",
    "serotonin",
    "chromatin",
    "macronutrient",
    "chronic pain",
    "polygenic",
    "gene",
    "caffeine sensitivity",
    "parathyroid",
    "vision-check",
    "metabolic rate",
}


MENTAL_HEALTH_KEYWORDS = {
    "depression",
    "burnout",
    "hypomania",
    "hysteria",
    "psychoticism",
    "identity diffusion",
    "acculturative stress",
    "sense-of-coherence score",
}


SPIRITUAL_KEYWORDS = {
    "spiritual",
    "religious",
    "sufi",
    "quran",
    "bahá",
    "reiki",
    "astrology",
    "i ching",
    "gnostic",
    "jewish",
    "hindu",
    "mantra",
    "kabbalah",
    "islamic",
    "sikh",
    "buddhist",
    "scripture",
    "sacred text",
    "shabbat",
    "dhikr",
    "yoga",
    "pilgrimage",
    "holiness",
    "satya",
}


COMMUNICATION_KEYWORDS = {
    "talkativeness",
    "brevity",
    "language use",
    "sentence structure",
    "storytelling",
    "outspokenness",
    "frankness",
    "listening",
    "non-verbal communication",
    "spelling",
    "eye-contact",
}


SKILL_KEYWORDS = {
    "reasoning",
    "knowledge",
    "skills",
    "ability",
    "analysis",
    "calculations",
    "memory",
    "perception",
    "understanding",
    "comprehension",
    "filing",
    "troubleshooting",
    "computer skills",
    "robotics",
    "mathematical",
    "data analysis",
    "information retention",
}


PREFERENCE_KEYWORDS = {
    "preference",
    "preferred",
    "attitude",
    "orientation",
    "desire",
    "motivation",
    "needs",
    "affinity",
}


EXTERNAL_FACT_KEYWORDS = {
    "count",
    "hours",
    "years",
    "frequency",
    "km/week",
    "time/day",
    "months",
    "subscribers",
    "subscription",
    "passport",
    "nationality",
    "attendance",
    "contributions",
    "usage",
    "intake",
    "commute",
    "presence",
    "ratio",
    "stamps",
    "sessions",
    "per day",
    "per week",
}


# ============================================================
# 2. NORMALIZATION
# ============================================================

def normalize_facet(value):
    """
    Creates a normalized version of the facet while preserving
    the original value separately.
    """

    value = str(value).strip()

    # Remove leading numeric IDs such as:
    # "800. Sufi practice: ..."
    value = re.sub(r"^\d+\.\s*", "", value)

    # Replace special dashes
    value = value.replace("–", "-")
    value = value.replace("—", "-")

    # Split camel-case words:
    # SelfEsteem -> Self Esteem
    value = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", value)

    # Remove trailing punctuation used by category headers
    value = re.sub(r"[:;]+$", "", value)

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    return value.strip().lower()


# ============================================================
# 3. HEADER / MALFORMED ENTRY DETECTION
# ============================================================

def is_header_like(raw_facet, normalized_facet):

    raw_facet = raw_facet.strip()

    # Strong signal: category headings frequently end with ":"
    if raw_facet.endswith(":"):
        return True

    # Detect common category/header terminology
    for term in HEADER_TERMS:
        if term in normalized_facet:
            return True

    return False


# ============================================================
# 4. FACET TYPE CLASSIFICATION
# ============================================================

def classify_facet(normalized_facet, header_like):

    if header_like:
        return "header_or_malformed"

    if any(keyword in normalized_facet for keyword in MEDICAL_KEYWORDS):
        return "medical_or_biological"

    if any(keyword in normalized_facet for keyword in MENTAL_HEALTH_KEYWORDS):
        return "psychological_or_mental_health"

    if any(keyword in normalized_facet for keyword in SPIRITUAL_KEYWORDS):
        return "spiritual_or_religious_practice"

    if any(keyword in normalized_facet for keyword in COMMUNICATION_KEYWORDS):
        return "communication_or_interaction"

    if any(keyword in normalized_facet for keyword in SKILL_KEYWORDS):
        return "skill_or_ability"

    if any(keyword in normalized_facet for keyword in PREFERENCE_KEYWORDS):
        return "preference_or_attitude"

    if any(keyword in normalized_facet for keyword in EXTERNAL_FACT_KEYWORDS):
        return "biographical_or_external_fact"

    # Default category for traits and behaviours
    return "behavioral_or_personality"


# ============================================================
# 5. OBSERVABILITY
# ============================================================

def assign_observability(facet_type):

    if facet_type == "header_or_malformed":
        return "not_observable"

    if facet_type == "medical_or_biological":
        return "not_observable"

    if facet_type == "psychological_or_mental_health":
        return "not_observable"

    # These can only be evaluated when explicitly mentioned
    if facet_type == "biographical_or_external_fact":
        return "conditional"

    if facet_type == "spiritual_or_religious_practice":
        return "conditional"

    # For traits, skills and preferences:
    # score only if sufficient evidence exists
    return "conditional"


# ============================================================
# 6. SENSITIVITY
# ============================================================

def assign_sensitivity(facet_type, normalized_facet):

    high_sensitivity_terms = {
        "drug-use",
        "physical-violence",
        "kink",
        "nationality",
        "cultural identity",
        "childhood",
        "attachment",
        "diagnosis",
        "chronic pain",
    }

    if facet_type in {
        "medical_or_biological",
        "psychological_or_mental_health"
    }:
        return "high"

    if any(term in normalized_facet for term in high_sensitivity_terms):
        return "high"

    if facet_type in {
        "spiritual_or_religious_practice",
        "biographical_or_external_fact"
    }:
        return "medium"

    return "low"


# ============================================================
# 7. ABSTENTION REASONS
# ============================================================

def get_abstention_reason(facet_type, observability):

    if facet_type == "header_or_malformed":
        return (
            "Header-like or malformed catalogue entry; "
            "excluded from retrieval and scoring."
        )

    if facet_type == "medical_or_biological":
        return (
            "Requires medical, laboratory, genetic, diagnostic, "
            "or other external evidence."
        )

    if facet_type == "psychological_or_mental_health":
        return (
            "Should not be diagnosed or assigned from ordinary "
            "conversational text alone."
        )

    if observability == "conditional":
        return (
            "Score only when the conversation provides direct, "
            "specific evidence about the speaker."
        )

    return "Insufficient conversational evidence."


# ============================================================
# 8. SCORING DEFINITIONS
# ============================================================

def get_scoring_definition(facet_type, observability):

    if observability == "not_observable":
        return None

    if facet_type == "communication_or_interaction":
        return (
            "Use a 1-5 ordinal scale based only on directly observable "
            "conversational evidence. Abstain when the sample is too "
            "short or unrepresentative."
        )

    if facet_type == "skill_or_ability":
        return (
            "Use a 1-5 ordinal scale based on demonstrated or explicitly "
            "described ability. Do not infer competence from interest alone."
        )

    if facet_type == "preference_or_attitude":
        return (
            "Use a 1-5 ordinal scale based on explicit statements or "
            "repeated direct evidence. Abstain when preference is unclear."
        )

    return (
        "Use a 1-5 ordinal scale based on direct conversational evidence. "
        "Do not infer stable traits from isolated statements."
    )


# ============================================================
# 9. MAIN PREPROCESSING FUNCTION
# ============================================================

def preprocess_facets(df):

    processed_rows = []

    for index, value in enumerate(df["Facets"], start=1):

        raw_facet = "" if pd.isna(value) else str(value).strip()

        normalized_facet = normalize_facet(raw_facet)

        is_empty = normalized_facet == ""

        if is_empty:
            header_like = True
            facet_type = "header_or_malformed"
        else:
            header_like = is_header_like(
                raw_facet,
                normalized_facet
            )

            facet_type = classify_facet(
                normalized_facet,
                header_like
            )

        observability = assign_observability(
            facet_type
        )

        sensitivity = assign_sensitivity(
            facet_type,
            normalized_facet
        )

        retrieval_enabled = (
            not header_like
            and observability != "not_observable"
        )

        scoring_definition = get_scoring_definition(
            facet_type,
            observability
        )

        abstention_reason = get_abstention_reason(
            facet_type,
            observability
        )

        processed_rows.append({
            "facet_id": index,
            "raw_facet": raw_facet,
            "normalized_facet": normalized_facet,
            "facet_type": facet_type,
            "conversation_observable": observability,
            "sensitivity": sensitivity,
            "is_header_like": header_like,
            "retrieval_enabled": retrieval_enabled,
            "scoring_definition": scoring_definition,
            "abstention_reason": abstention_reason,
        })

    return pd.DataFrame(processed_rows)


# ============================================================
# 10. LOAD CSV
# ============================================================

INPUT_FILE = '/content/Facets Assignment.csv'

df = pd.read_csv(INPUT_FILE)

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 11. RUN PREPROCESSING
# ============================================================

enriched_df = preprocess_facets(df)


# ============================================================
# 12. SAVE OUTPUT
# ============================================================

OUTPUT_FILE = "enriched_facets.csv"

enriched_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 13. AUDIT SUMMARY
# ============================================================

audit_summary = {
    "total_rows": int(len(enriched_df)),
    "header_like_rows": int(
        enriched_df["is_header_like"].sum()
    ),
    "retrieval_enabled_rows": int(
        enriched_df["retrieval_enabled"].sum()
    ),
    "facet_type_counts": (
        enriched_df["facet_type"]
        .value_counts()
        .to_dict()
    ),
    "observability_counts": (
        enriched_df["conversation_observable"]
        .value_counts()
        .to_dict()
    ),
    "sensitivity_counts": (
        enriched_df["sensitivity"]
        .value_counts()
        .to_dict()
    ),
}


print("\n" + "=" * 70)
print("FACET AUDIT SUMMARY")
print("=" * 70)

print(json.dumps(
    audit_summary,
    indent=4
))


# ============================================================
# 14. SAVE AUDIT SUMMARY
# ============================================================

with open(
    "audit_summary.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        audit_summary,
        file,
        indent=4
    )


# ============================================================
# 15. DISPLAY SAMPLE OUTPUT
# ============================================================

print("\nSample of enriched dataset:\n")

display(
    enriched_df.head(20)
)


print("\nFiles created:")
print("-", OUTPUT_FILE)
print("-", "audit_summary.json")

Original dataset shape: (399, 1)

Columns:
['Facets']

FACET AUDIT SUMMARY
{
    "total_rows": 399,
    "header_like_rows": 31,
    "retrieval_enabled_rows": 341,
    "facet_type_counts": {
        "behavioral_or_personality": 219,
        "spiritual_or_religious_practice": 35,
        "skill_or_ability": 32,
        "biographical_or_external_fact": 32,
        "header_or_malformed": 31,
        "medical_or_biological": 17,
        "preference_or_attitude": 12,
        "communication_or_interaction": 11,
        "psychological_or_mental_health": 10
    },
    "observability_counts": {
        "conditional": 341,
        "not_observable": 58
    },
    "sensitivity_counts": {
        "low": 298,
        "medium": 66,
        "high": 35
    }
}

Sample of enriched dataset:



,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity,is_header_like,retrieval_enabled,scoring_definition,abstention_reason
0,1,Risktaking,risktaking,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
1,2,Naivety,naivety,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
2,3,Acidity,acidity,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
3,4,Democratic Leadership:,democratic leadership,header_or_malformed,not_observable,low,True,False,None,Header-like or malformed catalogue entry; excl...
4,5,Common-sense,common-sense,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
5,6,Hesitation,hesitation,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
6,7,Discontentment,discontentment,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
7,8,Overprotectiveness,overprotectiveness,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
8,9,Merriness,merriness,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
9,10,Emotionalism,emotionalism,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...



Files created:
- enriched_facets.csv
- audit_summary.json


In [2]:
import re


# ============================================================
# HELPER: SAFE KEYWORD / PHRASE MATCHING
# ============================================================

def contains_keyword(text, keyword):
    """
    Matches complete words or complete phrases.

    Examples:
    contains_keyword("macronutrient ratio", "ratio")
    -> True

    contains_keyword("desperation", "ratio")
    -> False
    """

    pattern = r"(?<!\w)" + re.escape(keyword) + r"(?!\w)"

    return bool(
        re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )
    )


def contains_any_keyword(text, keywords):

    return any(
        contains_keyword(text, keyword)
        for keyword in keywords
    )


# ============================================================
# IMPROVED HEADER DETECTION
# ============================================================

def is_header_like(raw_facet, normalized_facet):

    raw_facet = raw_facet.strip()

    # A trailing colon is a strong signal of a catalogue heading.
    if raw_facet.endswith(":"):
        return True

    # Match complete words/phrases only.
    return contains_any_keyword(
        normalized_facet,
        HEADER_TERMS
    )


# ============================================================
# IMPROVED FACET CLASSIFICATION
# ============================================================

def classify_facet(normalized_facet, header_like):

    if header_like:
        return "header_or_malformed"

    if contains_any_keyword(
        normalized_facet,
        MEDICAL_KEYWORDS
    ):
        return "medical_or_biological"

    if contains_any_keyword(
        normalized_facet,
        MENTAL_HEALTH_KEYWORDS
    ):
        return "psychological_or_mental_health"

    if contains_any_keyword(
        normalized_facet,
        SPIRITUAL_KEYWORDS
    ):
        return "spiritual_or_religious_practice"

    if contains_any_keyword(
        normalized_facet,
        COMMUNICATION_KEYWORDS
    ):
        return "communication_or_interaction"

    if contains_any_keyword(
        normalized_facet,
        SKILL_KEYWORDS
    ):
        return "skill_or_ability"

    if contains_any_keyword(
        normalized_facet,
        PREFERENCE_KEYWORDS
    ):
        return "preference_or_attitude"

    if contains_any_keyword(
        normalized_facet,
        EXTERNAL_FACT_KEYWORDS
    ):
        return "biographical_or_external_fact"

    return "behavioral_or_personality"


# ============================================================
# RERUN PREPROCESSING
# ============================================================

enriched_df = preprocess_facets(df)

enriched_df.to_csv(
    "enriched_facets_v2.csv",
    index=False
)


# ============================================================
# CHECK THE PREVIOUS BUG
# ============================================================

display(
    enriched_df[
        enriched_df["raw_facet"]
        .str.contains(
            "Desperation",
            case=False,
            na=False
        )
    ]
)

,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity,is_header_like,retrieval_enabled,scoring_definition,abstention_reason
33,34,Desperation,desperation,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...


In [3]:
# ============================================================
# FULL AUDIT INSPECTION
# ============================================================

print("=" * 70)
print("FACET TYPE COUNTS")
print("=" * 70)

print(
    enriched_df["facet_type"]
    .value_counts()
)


print("\n" + "=" * 70)
print("OBSERVABILITY COUNTS")
print("=" * 70)

print(
    enriched_df["conversation_observable"]
    .value_counts()
)


print("\n" + "=" * 70)
print("SENSITIVITY COUNTS")
print("=" * 70)

print(
    enriched_df["sensitivity"]
    .value_counts()
)


print("\n" + "=" * 70)
print("HEADER-LIKE ROWS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["is_header_like"] == True
    ][
        [
            "facet_id",
            "raw_facet",
            "normalized_facet",
            "facet_type"
        ]
    ]
)


print("\n" + "=" * 70)
print("MEDICAL / BIOLOGICAL FACETS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["facet_type"]
        == "medical_or_biological"
    ][
        [
            "facet_id",
            "raw_facet",
            "conversation_observable",
            "retrieval_enabled"
        ]
    ]
)


print("\n" + "=" * 70)
print("PSYCHOLOGICAL / MENTAL HEALTH FACETS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["facet_type"]
        == "psychological_or_mental_health"
    ][
        [
            "facet_id",
            "raw_facet",
            "conversation_observable",
            "retrieval_enabled"
        ]
    ]
)


print("\n" + "=" * 70)
print("SPIRITUAL / RELIGIOUS FACETS")
print("=" * 70)

display(
    enriched_df[
        enriched_df["facet_type"]
        == "spiritual_or_religious_practice"
    ][
        [
            "facet_id",
            "raw_facet",
            "conversation_observable",
            "sensitivity"
        ]
    ]
)


print("\n" + "=" * 70)
print("ROWS EXCLUDED FROM RETRIEVAL")
print("=" * 70)

display(
    enriched_df[
        enriched_df["retrieval_enabled"] == False
    ][
        [
            "facet_id",
            "raw_facet",
            "facet_type",
            "abstention_reason"
        ]
    ]
)

FACET TYPE COUNTS
facet_type
behavioral_or_personality          228
spiritual_or_religious_practice     34
header_or_malformed                 31
biographical_or_external_fact       29
skill_or_ability                    28
medical_or_biological               15
preference_or_attitude              13
communication_or_interaction        11
psychological_or_mental_health      10
Name: count, dtype: int64

OBSERVABILITY COUNTS
conversation_observable
conditional       343
not_observable     56
Name: count, dtype: int64

SENSITIVITY COUNTS
sensitivity
low       304
medium     62
high       33
Name: count, dtype: int64

HEADER-LIKE ROWS


,facet_id,raw_facet,normalized_facet,facet_type
3,4,Democratic Leadership:,democratic leadership,header_or_malformed
21,22,HonestyHumility:,honesty humility,header_or_malformed
22,23,Relationship Building Themes:,relationship building themes,header_or_malformed
24,25,Numerical Reasoning Subcomponents:,numerical reasoning subcomponents,header_or_malformed
34,35,Affiliation Motivation:,affiliation motivation,header_or_malformed
48,49,Innovation and Creativity Components:,innovation and creativity components,header_or_malformed
49,50,Achievement Motivation:,achievement motivation,header_or_malformed
69,70,Work Styles,work styles,header_or_malformed
85,86,Leadership Potential:,leadership potential,header_or_malformed
90,91,Listening Skills Subcomponents:,listening skills subcomponents,header_or_malformed



MEDICAL / BIOLOGICAL FACETS


,facet_id,raw_facet,conversation_observable,retrieval_enabled
31,32,FSH level,not_observable,False
89,90,Sleep-disorder diagnosis,not_observable,False
124,125,Parathyroid-hormone level,not_observable,False
135,136,Chromatin-accessibility score,not_observable,False
149,150,Serotonin transporter availability,not_observable,False
162,163,Vision-check frequency,not_observable,False
164,165,Macronutrient ratio: fat,not_observable,False
229,230,Metabolic Rate (Low or High),not_observable,False
237,238,Macronutrient ratio: carbs,not_observable,False
244,245,Immune-response age,not_observable,False



PSYCHOLOGICAL / MENTAL HEALTH FACETS


,facet_id,raw_facet,conversation_observable,retrieval_enabled
57,58,Sense-of-coherence score,not_observable,False
60,61,Depression Symptoms,not_observable,False
83,84,Depression: Feelings of sadness and hopelessness,not_observable,False
102,103,Burnout Symptoms,not_observable,False
111,112,Psychoticism,not_observable,False
174,175,Depression (DEP),not_observable,False
176,177,Hypomania (Ma),not_observable,False
198,199,Psychological construct: Acculturative Stress ...,not_observable,False
201,202,Hysteria (Hy),not_observable,False
343,344,Psychological construct: Identity Diffusion level,not_observable,False



SPIRITUAL / RELIGIOUS FACETS


,facet_id,raw_facet,conversation_observable,sensitivity
37,38,Presence of Spiritual Pain,conditional,medium
42,43,Pilgrimage participation count,conditional,medium
103,104,Holiness,conditional,medium
134,135,800. Sufi practice: Sufi retreat attendance count,conditional,medium
136,137,644. Spiritual virtue: Humility practice index,conditional,medium
137,138,754. I Ching hexagram 36 resonance level,conditional,medium
142,143,596. Religious practice: Quran khatam cycles p...,conditional,medium
143,144,926. Bahá’í spiritual metric: Ridván festival ...,conditional,medium
156,157,607. Energy-healing practice: Reiki sessions /...,conditional,medium
157,158,692. Astrology: Rising sign is Scorpio,conditional,medium



ROWS EXCLUDED FROM RETRIEVAL


,facet_id,raw_facet,facet_type,abstention_reason
3,4,Democratic Leadership:,header_or_malformed,Header-like or malformed catalogue entry; excl...
21,22,HonestyHumility:,header_or_malformed,Header-like or malformed catalogue entry; excl...
22,23,Relationship Building Themes:,header_or_malformed,Header-like or malformed catalogue entry; excl...
24,25,Numerical Reasoning Subcomponents:,header_or_malformed,Header-like or malformed catalogue entry; excl...
31,32,FSH level,medical_or_biological,"Requires medical, laboratory, genetic, diagnos..."
34,35,Affiliation Motivation:,header_or_malformed,Header-like or malformed catalogue entry; excl...
48,49,Innovation and Creativity Components:,header_or_malformed,Header-like or malformed catalogue entry; excl...
49,50,Achievement Motivation:,header_or_malformed,Header-like or malformed catalogue entry; excl...
57,58,Sense-of-coherence score,psychological_or_mental_health,Should not be diagnosed or assigned from ordin...
60,61,Depression Symptoms,psychological_or_mental_health,Should not be diagnosed or assigned from ordin...


In [4]:
# ============================================================
# FINAL HEADER AUDIT
# Check header-like rows that do NOT end with a colon.
# These are the rows most likely to reveal false positives.
# ============================================================

non_colon_headers = enriched_df[
    (enriched_df["is_header_like"] == True)
    &
    (~enriched_df["raw_facet"].str.strip().str.endswith(":"))
]

print("HEADER-LIKE ROWS WITHOUT A TRAILING COLON:")
print("Count:", len(non_colon_headers))

display(
    non_colon_headers[
        [
            "facet_id",
            "raw_facet",
            "normalized_facet",
            "facet_type"
        ]
    ]
)

HEADER-LIKE ROWS WITHOUT A TRAILING COLON:
Count: 1


,facet_id,raw_facet,normalized_facet,facet_type
69,70,Work Styles,work styles,header_or_malformed


In [5]:
!pip -q install sentence-transformers scikit-learn

In [15]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [16]:
# Load the enriched facet catalogue
enriched_df = pd.read_csv("/content/enriched_facets.csv")

print("Total facets:", len(enriched_df))
print("Columns:")
print(enriched_df.columns.tolist())

display(enriched_df.head())

Total facets: 399
Columns:
['facet_id', 'raw_facet', 'normalized_facet', 'facet_type', 'conversation_observable', 'sensitivity', 'is_header_like', 'retrieval_enabled', 'scoring_definition', 'abstention_reason']


,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity,is_header_like,retrieval_enabled,scoring_definition,abstention_reason
0,1,Risktaking,risktaking,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
1,2,Naivety,naivety,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
2,3,Acidity,acidity,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...
3,4,Democratic Leadership:,democratic leadership,header_or_malformed,not_observable,low,True,False,NaN,Header-like or malformed catalogue entry; excl...
4,5,Common-sense,common-sense,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...


In [17]:
# Keep only facets approved for retrieval
retrieval_df = enriched_df[
    enriched_df["retrieval_enabled"] == True
].copy()

print("Total facets:", len(enriched_df))
print("Retrieval-enabled facets:", len(retrieval_df))
print("Excluded from retrieval:", len(enriched_df) - len(retrieval_df))

display(
    retrieval_df[
        [
            "facet_id",
            "raw_facet",
            "normalized_facet",
            "facet_type",
            "conversation_observable",
            "sensitivity"
        ]
    ].head(10)
)

Total facets: 399
Retrieval-enabled facets: 343
Excluded from retrieval: 56


,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity
0,1,Risktaking,risktaking,behavioral_or_personality,conditional,low
1,2,Naivety,naivety,behavioral_or_personality,conditional,low
2,3,Acidity,acidity,behavioral_or_personality,conditional,low
4,5,Common-sense,common-sense,behavioral_or_personality,conditional,low
5,6,Hesitation,hesitation,behavioral_or_personality,conditional,low
6,7,Discontentment,discontentment,behavioral_or_personality,conditional,low
7,8,Overprotectiveness,overprotectiveness,behavioral_or_personality,conditional,low
8,9,Merriness,merriness,behavioral_or_personality,conditional,low
9,10,Emotionalism,emotionalism,behavioral_or_personality,conditional,low
10,11,Self-improvement,self-improvement,behavioral_or_personality,conditional,low


In [18]:
# Create a richer text representation for each retrieval-enabled facet

def build_facet_document(row):
    parts = [
        f"Facet: {row['normalized_facet']}",
        f"Category: {row['facet_type']}",
        f"Observability: {row['conversation_observable']}",
        f"Scoring guidance: {row['scoring_definition']}"
    ]

    return " | ".join(
        str(part)
        for part in parts
        if pd.notna(part)
    )


retrieval_df["facet_document"] = retrieval_df.apply(
    build_facet_document,
    axis=1
)

print("Sample retrieval documents:\n")

for i, text in enumerate(retrieval_df["facet_document"].head(5)):
    print(f"{i + 1}. {text}\n")

Sample retrieval documents:

1. Facet: risktaking | Category: behavioral_or_personality | Observability: conditional | Scoring guidance: Use a 1-5 ordinal scale based on direct conversational evidence. Do not infer stable traits from isolated statements.

2. Facet: naivety | Category: behavioral_or_personality | Observability: conditional | Scoring guidance: Use a 1-5 ordinal scale based on direct conversational evidence. Do not infer stable traits from isolated statements.

3. Facet: acidity | Category: behavioral_or_personality | Observability: conditional | Scoring guidance: Use a 1-5 ordinal scale based on direct conversational evidence. Do not infer stable traits from isolated statements.

4. Facet: common-sense | Category: behavioral_or_personality | Observability: conditional | Scoring guidance: Use a 1-5 ordinal scale based on direct conversational evidence. Do not infer stable traits from isolated statements.

5. Facet: hesitation | Category: behavioral_or_personality | Observ

In [19]:
# Build TF-IDF representation of the facet documents

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    retrieval_df["facet_document"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF matrix shape: (343, 1681)
Vocabulary size: 1681


In [20]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_facets(query, top_k=5):
    """
    Retrieve the most relevant facets for a natural-language query.
    """

    # Convert the query into the same TF-IDF vector space
    query_vector = vectorizer.transform([query])

    # Calculate cosine similarity against all facets
    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Get indices of the top matching facets
    top_indices = similarities.argsort()[::-1][:top_k]

    # Create results dataframe
    results = retrieval_df.iloc[top_indices].copy()

    # Add similarity scores
    results["similarity_score"] = similarities[top_indices]

    return results[
        [
            "facet_id",
            "raw_facet",
            "normalized_facet",
            "facet_type",
            "sensitivity",
            "similarity_score"
        ]
    ]

In [21]:
query = "I enjoy taking risks and trying new things"

results = retrieve_facets(query, top_k=5)

display(results)

NameError: name 'vectorizer' is not defined

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

tfidf_matrix = vectorizer.fit_transform(
    retrieval_df["normalized_facet"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (343, 581)
Vocabulary size: 581


In [23]:
results = retrieve_facets(
    "I enjoy taking risks and trying new things",
    top_k=5
)

display(results)

,facet_id,raw_facet,normalized_facet,facet_type,sensitivity,similarity_score
65,66,Creative risk-taking tendency,creative risk-taking tendency,behavioral_or_personality,low,0.377620
335,336,945. New-Age spiritual metric: Channeling sess...,new-age spiritual metric: channeling sessions ...,spiritual_or_religious_practice,medium,0.301274
129,130,Psychological construct: Need for Achievement ...,psychological construct: need for achievement ...,behavioral_or_personality,low,0.000000
130,131,Cantankerousness,cantankerousness,behavioral_or_personality,low,0.000000
131,132,Material properties knowledge,material properties knowledge,skill_or_ability,low,0.000000


In [24]:
def retrieve_facets(query, top_k=5):

    # Convert query into the TF-IDF vector space
    query_vector = vectorizer.transform([query])

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Add scores to a copy of retrieval data
    results = retrieval_df.copy()
    results["similarity_score"] = similarity_scores

    # Remove completely irrelevant results
    results = results[
        results["similarity_score"] > 0
    ]

    # Sort by similarity
    results = results.sort_values(
        by="similarity_score",
        ascending=False
    )

    # Return top results
    return results.head(top_k)

In [25]:
query = "I enjoy taking risks and trying new things"

results = retrieve_facets(query, top_k=5)

display(results)

,facet_id,raw_facet,normalized_facet,facet_type,conversation_observable,sensitivity,is_header_like,retrieval_enabled,scoring_definition,abstention_reason,facet_document,similarity_score
65,66,Creative risk-taking tendency,creative risk-taking tendency,behavioral_or_personality,conditional,low,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...,Facet: creative risk-taking tendency | Categor...,0.377620
335,336,945. New-Age spiritual metric: Channeling sess...,new-age spiritual metric: channeling sessions ...,spiritual_or_religious_practice,conditional,medium,False,True,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...,Facet: new-age spiritual metric: channeling se...,0.301274


In [26]:
test_queries = [
    "I enjoy taking risks and trying new things",
    "I often hesitate before making important decisions",
    "I get easily irritated and annoyed by people",
    "I constantly try to improve myself"
]

for query in test_queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_facets(query, top_k=3)

    display(
        results[
            [
                "facet_id",
                "raw_facet",
                "facet_type",
                "sensitivity",
                "similarity_score"
            ]
        ]
    )


QUERY: I enjoy taking risks and trying new things


,facet_id,raw_facet,facet_type,sensitivity,similarity_score
65,66,Creative risk-taking tendency,behavioral_or_personality,low,0.377620
335,336,945. New-Age spiritual metric: Channeling sess...,spiritual_or_religious_practice,medium,0.301274



QUERY: I often hesitate before making important decisions


,facet_id,raw_facet,facet_type,sensitivity,similarity_score
299,300,Decision-making speed,behavioral_or_personality,low,0.551153
56,57,Decision-making decisiveness,behavioral_or_personality,low,0.536973
374,375,Decision-Making Confidence,behavioral_or_personality,low,0.536973



QUERY: I get easily irritated and annoyed by people


,facet_id,raw_facet,facet_type,sensitivity,similarity_score



QUERY: I constantly try to improve myself


,facet_id,raw_facet,facet_type,sensitivity,similarity_score


## Phase 2 Evaluation: TF-IDF Retrieval Baseline

The TF-IDF retrieval system was tested using multiple natural-language queries.

### Successful retrieval examples

- Risk-taking queries successfully retrieved **Creative risk-taking tendency**.
- Decision-making queries successfully retrieved relevant facets including:
  - Decision-making speed
  - Decision-making decisiveness
  - Decision-Making Confidence

### Observed limitations

The system did not retrieve relevant facets for some queries expressed using indirect language or synonyms, such as:

- "I get easily irritated and annoyed by people"
- "I constantly try to improve myself"

This demonstrates a limitation of TF-IDF retrieval: it primarily relies on lexical token overlap rather than understanding semantic meaning.

### Conclusion

TF-IDF provides a useful lexical retrieval baseline. However, semantic embedding-based retrieval will be explored next to improve handling of paraphrasing, synonyms, and meaning-based queries.

In [27]:
# ============================================================
# Phase 3: Semantic Embedding Retrieval
# ============================================================

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    embedding_model_name
)

print("Loaded embedding model:", embedding_model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [28]:
# ============================================================
# Create embeddings for the retrieval-enabled facet corpus
# ============================================================

facet_texts = retrieval_df["facet_document"].tolist()

facet_embeddings = embedding_model.encode(
    facet_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of facet embeddings:", len(facet_embeddings))
print("Embedding matrix shape:", facet_embeddings.shape)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Number of facet embeddings: 343
Embedding matrix shape: (343, 384)


In [29]:
# ============================================================
# Semantic Retrieval Function
# ============================================================

def retrieve_facets_semantic(query, top_k=5):
    """
    Retrieve the most semantically relevant facets for a natural-language query.
    """

    # Convert the query into a normalized semantic embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Since embeddings are normalized, dot product = cosine similarity
    similarity_scores = facet_embeddings @ query_embedding

    # Add similarity scores to retrieval dataframe
    results = retrieval_df.copy()
    results["semantic_similarity"] = similarity_scores

    # Sort by semantic similarity
    results = results.sort_values(
        by="semantic_similarity",
        ascending=False
    )

    # Return top-k results
    return results.head(top_k)[
        [
            "facet_id",
            "raw_facet",
            "facet_type",
            "sensitivity",
            "semantic_similarity"
        ]
    ]

In [30]:
semantic_test_queries = [
    "I get easily irritated and annoyed by people",
    "I constantly try to improve myself"
]

for query in semantic_test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_facets_semantic(
        query,
        top_k=5
    )

    display(results)


QUERY: I get easily irritated and annoyed by people


,facet_id,raw_facet,facet_type,sensitivity,semantic_similarity
360,361,Irritability,behavioral_or_personality,low,0.320779
388,389,Patience: Resistance to anger,behavioral_or_personality,low,0.293809
132,133,Hostility,behavioral_or_personality,low,0.275531
6,7,Discontentment,behavioral_or_personality,low,0.268080
233,234,Boredom Susceptibility,behavioral_or_personality,low,0.259929



QUERY: I constantly try to improve myself


,facet_id,raw_facet,facet_type,sensitivity,semantic_similarity
10,11,Self-improvement,behavioral_or_personality,low,0.315223
173,174,565. Practice frequency: Walking meditation,biographical_or_external_fact,medium,0.230383
394,395,793. Sufi practice: Dhikr repetitions / day,spiritual_or_religious_practice,medium,0.223792
390,391,Self-Efficacy,behavioral_or_personality,low,0.220180
365,366,823. Buddhist practice: Eightfold Path – Right...,spiritual_or_religious_practice,medium,0.219540


In [31]:
# ============================================================
# Refined semantic corpus
# Use facet meaning, not repeated scoring boilerplate
# ============================================================

retrieval_df["semantic_document"] = (
    "Facet: " + retrieval_df["normalized_facet"].astype(str)
    + " | Category: "
    + retrieval_df["facet_type"].astype(str)
)

semantic_texts = retrieval_df["semantic_document"].tolist()

print("Sample semantic documents:\n")

for text in semantic_texts[:5]:
    print(text)

Sample semantic documents:

Facet: risktaking | Category: behavioral_or_personality
Facet: naivety | Category: behavioral_or_personality
Facet: acidity | Category: behavioral_or_personality
Facet: common-sense | Category: behavioral_or_personality
Facet: hesitation | Category: behavioral_or_personality


In [32]:
# ============================================================
# Refined semantic corpus
# Use human-readable facet names and minimal metadata
# ============================================================

retrieval_df["semantic_document"] = (
    "Facet: " + retrieval_df["raw_facet"].astype(str)
    + " | Category: "
    + retrieval_df["facet_type"].astype(str)
)

semantic_texts = retrieval_df["semantic_document"].tolist()

print("Sample semantic documents:\n")

for text in semantic_texts[:5]:
    print(text)

Sample semantic documents:

Facet: Risktaking | Category: behavioral_or_personality
Facet: Naivety | Category: behavioral_or_personality
Facet: Acidity | Category: behavioral_or_personality
Facet: Common-sense | Category: behavioral_or_personality
Facet: Hesitation | Category: behavioral_or_personality


In [33]:
# ============================================================
# Regenerate facet embeddings using refined semantic documents
# ============================================================

facet_embeddings_refined = embedding_model.encode(
    semantic_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of refined facet embeddings:", len(facet_embeddings_refined))
print("Refined embedding matrix shape:", facet_embeddings_refined.shape)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Number of refined facet embeddings: 343
Refined embedding matrix shape: (343, 384)


In [34]:
# ============================================================
# Semantic Retrieval - Refined Corpus
# ============================================================

def retrieve_facets_semantic_refined(query, top_k=5):
    """
    Retrieve the most semantically relevant facets using
    the refined facet-only semantic corpus.
    """

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Normalized vectors -> dot product equals cosine similarity
    similarity_scores = facet_embeddings_refined @ query_embedding

    results = retrieval_df.copy()
    results["semantic_similarity"] = similarity_scores

    results = results.sort_values(
        by="semantic_similarity",
        ascending=False
    )

    return results.head(top_k)[
        [
            "facet_id",
            "raw_facet",
            "facet_type",
            "sensitivity",
            "semantic_similarity"
        ]
    ]

In [35]:
semantic_test_queries = [
    "I get easily irritated and annoyed by people",
    "I constantly try to improve myself"
]

for query in semantic_test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_facets_semantic_refined(
        query,
        top_k=5
    )

    display(results)


QUERY: I get easily irritated and annoyed by people


,facet_id,raw_facet,facet_type,sensitivity,semantic_similarity
360,361,Irritability,behavioral_or_personality,low,0.503371
388,389,Patience: Resistance to anger,behavioral_or_personality,low,0.414215
132,133,Hostility,behavioral_or_personality,low,0.403421
6,7,Discontentment,behavioral_or_personality,low,0.386682
276,277,Clumsiness,behavioral_or_personality,low,0.378843



QUERY: I constantly try to improve myself


,facet_id,raw_facet,facet_type,sensitivity,semantic_similarity
10,11,Self-improvement,behavioral_or_personality,low,0.440691
390,391,Self-Efficacy,behavioral_or_personality,low,0.276437
389,390,Flawlessness,behavioral_or_personality,low,0.265159
348,349,Psychological construct: Perfectionistic Striv...,behavioral_or_personality,low,0.262784
167,168,Dance-style mastery diversity,behavioral_or_personality,low,0.251046


In [38]:
# ============================================================
# Hybrid Retrieval
# TF-IDF candidate retrieval + Semantic candidate retrieval
# ============================================================

def retrieve_facets_hybrid(query, lexical_k=5, semantic_k=5):
    """
    Retrieve candidate facets using both lexical TF-IDF and
    semantic embedding similarity.

    The two retrieval methods are combined using candidate union
    rather than directly averaging their scores.
    """

    # ----------------------------
    # 1. Lexical retrieval
    # ----------------------------
    query_vector = vectorizer.transform([query])

    lexical_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    lexical_indices = np.argsort(
        lexical_scores
    )[::-1][:lexical_k]

    # Keep only positive lexical matches
    lexical_indices = [
        int(idx)
        for idx in lexical_indices
        if lexical_scores[idx] > 0
    ]

    # ----------------------------
    # 2. Semantic retrieval
    # ----------------------------
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    semantic_scores = (
        facet_embeddings_refined @ query_embedding
    )

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:semantic_k].tolist()

    # ----------------------------
    # 3. Candidate union
    # ----------------------------
    candidate_indices = list(
        dict.fromkeys(
            lexical_indices + semantic_indices
        )
    )

    results = retrieval_df.iloc[
        candidate_indices
    ].copy()

    # Add both similarity scores
    results["lexical_similarity"] = [
        lexical_scores[idx]
        for idx in candidate_indices
    ]

    results["semantic_similarity"] = [
        semantic_scores[idx]
        for idx in candidate_indices
    ]

    # Mark retrieval source
    results["retrieval_source"] = [
        (
            "hybrid"
            if idx in lexical_indices and idx in semantic_indices
            else "lexical"
            if idx in lexical_indices
            else "semantic"
        )
        for idx in candidate_indices
    ]

    # Sort primarily by semantic relevance
    results = results.sort_values(
        by="semantic_similarity",
        ascending=False
    )

    return results[
        [
            "facet_id",
            "raw_facet",
            "facet_type",
            "sensitivity",
            "retrieval_source",
            "lexical_similarity",
            "semantic_similarity"
        ]
    ]

In [39]:
hybrid_test_queries = [
    "I enjoy taking risks and trying new things",
    "I often hesitate before making important decisions",
    "I get easily irritated and annoyed by people",
    "I constantly try to improve myself"
]

for query in hybrid_test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    results = retrieve_facets_hybrid(
        query,
        lexical_k=5,
        semantic_k=5
    )

    display(results)


QUERY: I enjoy taking risks and trying new things


,facet_id,raw_facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity
65,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.377620,0.461682
0,1,Risktaking,behavioral_or_personality,low,semantic,0.000000,0.444591
378,379,Creative resilience,behavioral_or_personality,low,semantic,0.000000,0.337595
14,15,Adventure-Seeking Behavior,behavioral_or_personality,low,semantic,0.000000,0.323838
266,267,Fearfulness: Fear of physical dangers,behavioral_or_personality,low,semantic,0.000000,0.321220
335,336,945. New-Age spiritual metric: Channeling sess...,spiritual_or_religious_practice,medium,lexical,0.301274,0.143511



QUERY: I often hesitate before making important decisions


,facet_id,raw_facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity
56,57,Decision-making decisiveness,behavioral_or_personality,low,hybrid,0.536973,0.318766
374,375,Decision-Making Confidence,behavioral_or_personality,low,hybrid,0.536973,0.287011
5,6,Hesitation,behavioral_or_personality,low,semantic,0.000000,0.264067
299,300,Decision-making speed,behavioral_or_personality,low,hybrid,0.551153,0.250518
65,66,Creative risk-taking tendency,behavioral_or_personality,low,semantic,0.000000,0.250233
300,301,Psychological construct: Excuse-Making tendency,behavioral_or_personality,low,lexical,0.471161,0.172323



QUERY: I get easily irritated and annoyed by people


,facet_id,raw_facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity
360,361,Irritability,behavioral_or_personality,low,semantic,0.0,0.503371
388,389,Patience: Resistance to anger,behavioral_or_personality,low,semantic,0.0,0.414215
132,133,Hostility,behavioral_or_personality,low,semantic,0.0,0.403421
6,7,Discontentment,behavioral_or_personality,low,semantic,0.0,0.386682
276,277,Clumsiness,behavioral_or_personality,low,semantic,0.0,0.378843



QUERY: I constantly try to improve myself


,facet_id,raw_facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity
10,11,Self-improvement,behavioral_or_personality,low,semantic,0.0,0.440691
390,391,Self-Efficacy,behavioral_or_personality,low,semantic,0.0,0.276437
389,390,Flawlessness,behavioral_or_personality,low,semantic,0.0,0.265159
348,349,Psychological construct: Perfectionistic Striv...,behavioral_or_personality,low,semantic,0.0,0.262784
167,168,Dance-style mastery diversity,behavioral_or_personality,low,semantic,0.0,0.251046


In [40]:
# ============================================================
# Retrieval Experiment Summary
# ============================================================

experiment_summary = {
    "total_facets": len(enriched_df),
    "retrieval_enabled_facets": len(retrieval_df),
    "tfidf_vocabulary_size": len(vectorizer.vocabulary_),
    "tfidf_matrix_shape": tfidf_matrix.shape,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_matrix_shape": facet_embeddings_refined.shape,
    "retrieval_strategy": "hybrid candidate union (TF-IDF + semantic embeddings)"
}

for key, value in experiment_summary.items():
    print(f"{key}: {value}")

total_facets: 399
retrieval_enabled_facets: 343
tfidf_vocabulary_size: 581
tfidf_matrix_shape: (343, 581)
embedding_model: sentence-transformers/all-MiniLM-L6-v2
embedding_matrix_shape: (343, 384)
retrieval_strategy: hybrid candidate union (TF-IDF + semantic embeddings)


In [41]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 8.5 MB/s eta 0:00:00


In [42]:
import torch

if torch.cuda.is_available():
    print("GPU available:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected")

No GPU detected


In [43]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.6 MB/s eta 0:00:00


In [44]:
import os
from getpass import getpass
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass(
    "Enter your Groq API key (input will be hidden): "
)

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq client initialized successfully.")

Enter your Groq API key (input will be hidden): ··········
Groq client initialized successfully.


In [45]:
models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound-mini
whisper-large-v3
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-20b
qwen/qwen3.6-27b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
groq/compound
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo
qwen/qwen3.8-27b
meta-llama/llama-prompt-guard-2-86m


In [46]:
SCORING_MODEL = "allam-2-7b"

response = client.chat.completions.create(
    model=SCORING_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a careful evidence-based evaluator. "
                "Only make judgments supported by the provided text."
            )
        },
        {
            "role": "user",
            "content": (
                "Conversation: I get easily irritated and annoyed by people.\n\n"
                "Facet: Irritability\n\n"
                "Is there direct evidence for this facet? "
                "Answer briefly."
            )
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

Yes, there is direct evidence for the facet "Irritability" in the provided conversation, as the statement "I get easily irritated and annoyed by people" describes the individual's experience of becoming frustrated or annoyed in response to others, which aligns with the definition of irritability. 


In [75]:
# ============================================================
# Direct-Evidence-Gated LLM Facet Scorer
# ============================================================

import json


def score_facet_with_llm(conversation, facet_row):

    facet_name = str(facet_row["raw_facet"])
    facet_id = int(facet_row["facet_id"])
    scoring_definition = str(facet_row["scoring_definition"])
    abstention_reason = str(facet_row["abstention_reason"])

    # ========================================================
    # STEP 1: EXTRACT AND VERIFY DIRECT EVIDENCE
    # ========================================================

    evidence_system_prompt = """
You are a strict evidence verifier.

Your task is NOT to score a personality trait.

You must determine whether the conversation contains a statement that
DIRECTLY and SPECIFICALLY describes the exact requested facet.

Follow these rules strictly:

1. First identify an exact statement made by the speaker.
2. Then determine whether that statement is directly about the EXACT facet.
3. Do not infer one trait from another.
4. Do not treat semantic similarity as direct evidence.
5. Do not infer stable traits from an isolated statement.
6. If the statement discusses a different trait, directly_about_facet
   MUST be false.
7. If uncertain, directly_about_facet MUST be false.

Examples:

Statement:
"I enjoy taking risks."

Facet:
"Risktaking"

directly_about_facet = true


Statement:
"I enjoy taking risks."

Facet:
"Creative resilience"

directly_about_facet = false


Statement:
"I enjoy taking risks."

Facet:
"Impudence"

directly_about_facet = false


Statement:
"I constantly try to improve myself."

Facet:
"Significance: Desire to make an impact"

directly_about_facet = false


Return valid JSON only:

{
    "evidence_found": true or false,
    "speaker_statement": "exact statement from conversation or null",
    "directly_about_facet": true or false,
    "reason": "brief explanation"
}
"""

    evidence_user_prompt = f"""
CONVERSATION:
{conversation}

EXACT FACET TO EVALUATE:
{facet_name}

SCORING GUIDANCE:
{scoring_definition}

ABSTENTION GUIDANCE:
{abstention_reason}

Your task:

1. Find an exact statement from the conversation that could be relevant.
2. Determine whether that statement DIRECTLY describes
   "{facet_name}".

A related trait is NOT sufficient.

Return JSON only.
"""

    evidence_response = client.chat.completions.create(
        model=SCORING_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": evidence_system_prompt
            },
            {
                "role": "user",
                "content": evidence_user_prompt
            }
        ]
    )

    # --------------------------------------------------------
    # Parse evidence result safely
    # --------------------------------------------------------

    try:
        evidence_result = json.loads(
            evidence_response.choices[0].message.content
        )

    except json.JSONDecodeError:

        evidence_result = {
            "evidence_found": False,
            "speaker_statement": None,
            "directly_about_facet": False,
            "reason": "Evidence verification output could not be parsed."
        }

    evidence_found = evidence_result.get("evidence_found") is True

    directly_about_facet = (
        evidence_result.get("directly_about_facet") is True
    )

    speaker_statement = evidence_result.get(
        "speaker_statement"
    )

    # ========================================================
    # DETERMINISTIC ABSTENTION GATE
    # ========================================================

    if (
        not evidence_found
        or not directly_about_facet
        or not isinstance(speaker_statement, str)
        or not speaker_statement.strip()
    ):

        return {
            "facet_id": facet_id,
            "status": "insufficient_evidence",
            "score": None,
            "confidence": 1.0,
            "evidence": evidence_result.get(
                "reason",
                "No direct, facet-specific evidence was found."
            )
        }

    # ========================================================
    # STEP 2: SCORE ONLY VERIFIED FACETS
    # ========================================================

    scoring_system_prompt = """
You are an evidence-based ordinal evaluator.

The evidence verification stage has determined that the supplied
speaker statement directly describes the requested facet.

Score the facet strictly according to the provided scoring definition.

Rules:

1. Use only the verified speaker statement as evidence.
2. Do not infer additional traits.
3. Do not use outside knowledge.
4. Score must be an integer from 1 to 5.
5. Do not return decimal scores.
6. Confidence must be between 0 and 1.

Return valid JSON only:

{
    "score": 1,
    "confidence": 0.0,
    "evidence": "brief explanation grounded in the verified statement"
}
"""

    scoring_user_prompt = f"""
FACET:
{facet_name}

SCORING DEFINITION:
{scoring_definition}

VERIFIED SPEAKER STATEMENT:
"{speaker_statement}"

Score the requested facet using only this verified statement.

Return JSON only.
"""

    scoring_response = client.chat.completions.create(
        model=SCORING_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": scoring_system_prompt
            },
            {
                "role": "user",
                "content": scoring_user_prompt
            }
        ]
    )

    # --------------------------------------------------------
    # Parse scoring result safely
    # --------------------------------------------------------

    try:

        score_result = json.loads(
            scoring_response.choices[0].message.content
        )

        return {
            "facet_id": facet_id,
            "status": "scored",
            "score": score_result.get("score"),
            "confidence": score_result.get("confidence"),
            "evidence": score_result.get(
                "evidence",
                f'Grounded in: "{speaker_statement}"'
            )
        }

    except json.JSONDecodeError:

        return {
            "facet_id": facet_id,
            "status": "parse_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "Scoring output could not be parsed."
        }

In [64]:
conversation = "I get easily irritated and annoyed by people."

irritability_row = retrieval_df[
    retrieval_df["raw_facet"] == "Irritability"
].iloc[0]

result = score_facet_with_llm(
    conversation,
    irritability_row
)

print(json.dumps(result, indent=2))

{
  "facet_id": 361,
  "status": "scored",
  "score": 3,
  "confidence": 0.5,
  "evidence": "The person mentions getting easily irritated and annoyed by people."
}


In [65]:
# ============================================================
# Structured Output Validation
# ============================================================

VALID_STATUSES = {
    "scored",
    "insufficient_evidence"
}


def validate_score_result(result, expected_facet_id):
    """
    Validate and normalize an LLM scoring result.

    Returns:
        validated_result, is_valid
    """

    # Check that the result is a dictionary
    if not isinstance(result, dict):
        return {
            "facet_id": expected_facet_id,
            "status": "validation_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "LLM output is not a JSON object."
        }, False

    # Validate facet_id
    if result.get("facet_id") != expected_facet_id:
        return {
            "facet_id": expected_facet_id,
            "status": "validation_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "Returned facet_id does not match requested facet."
        }, False

    # Validate status
    status = result.get("status")

    if status not in VALID_STATUSES:
        return {
            "facet_id": expected_facet_id,
            "status": "validation_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "Invalid status returned by model."
        }, False

    # Validate confidence
    confidence = result.get("confidence")

    if not isinstance(confidence, (int, float)):
        return {
            "facet_id": expected_facet_id,
            "status": "validation_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "Confidence is not numeric."
        }, False

    confidence = float(confidence)

    if not 0.0 <= confidence <= 1.0:
        return {
            "facet_id": expected_facet_id,
            "status": "validation_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "Confidence is outside the allowed range."
        }, False

    # Validate scored outputs
    if status == "scored":

        score = result.get("score")

        # bool is technically an int in Python, so exclude it
        if (
            isinstance(score, bool)
            or not isinstance(score, int)
            or score < 1
            or score > 5
        ):
            return {
                "facet_id": expected_facet_id,
                "status": "validation_error",
                "score": None,
                "confidence": 0.0,
                "evidence": "Scored result must contain an integer score from 1 to 5."
            }, False

    # Validate abstentions
    if status == "insufficient_evidence":

        if result.get("score") is not None:
            return {
                "facet_id": expected_facet_id,
                "status": "validation_error",
                "score": None,
                "confidence": 0.0,
                "evidence": "Abstained result must have score = null."
            }, False

    # Validate evidence
    evidence = result.get("evidence")

    if not isinstance(evidence, str) or not evidence.strip():
        return {
            "facet_id": expected_facet_id,
            "status": "validation_error",
            "score": None,
            "confidence": 0.0,
            "evidence": "Missing evidence explanation."
        }, False

    # Normalize confidence
    result["confidence"] = confidence

    return result, True

In [66]:
validated_result, is_valid = validate_score_result(
    result,
    expected_facet_id=361
)

print("Valid:", is_valid)
print(json.dumps(validated_result, indent=2))

Valid: True
{
  "facet_id": 361,
  "status": "scored",
  "score": 3,
  "confidence": 0.5,
  "evidence": "The person mentions getting easily irritated and annoyed by people."
}


In [67]:
# Show a few facets that are clearly unrelated to irritability
retrieval_df[
    ["facet_id", "raw_facet", "facet_type", "sensitivity"]
].sample(10, random_state=42)

,facet_id,raw_facet,facet_type,sensitivity
279,280,Robotics-interaction frequency,skill_or_ability,low
137,138,754. I Ching hexagram 36 resonance level,spiritual_or_religious_practice,medium
133,134,Independence,behavioral_or_personality,low
50,51,Feedback-giving frequency,biographical_or_external_fact,medium
147,148,Self-righteousness,behavioral_or_personality,low
320,321,Nationality,biographical_or_external_fact,high
214,215,Blissfulness,behavioral_or_personality,low
211,212,Outspokenness,communication_or_interaction,low
396,397,Doggedness,behavioral_or_personality,low
29,30,Compulsive activities,behavioral_or_personality,low


In [68]:
conversation = "I get easily irritated and annoyed by people."

unsupported_row = retrieval_df[
    retrieval_df["raw_facet"] == "Robotics-interaction frequency"
].iloc[0]

unsupported_result = score_facet_with_llm(
    conversation,
    unsupported_row
)

validated_unsupported_result, is_valid = validate_score_result(
    unsupported_result,
    expected_facet_id=int(unsupported_row["facet_id"])
)

print("Valid:", is_valid)
print(json.dumps(validated_unsupported_result, indent=2))

Valid: True
{
  "facet_id": 280,
  "status": "insufficient_evidence",
  "score": null,
  "confidence": 1.0,
  "evidence": "The conversation discusses the speaker's irritability and annoyance towards people, not their interaction frequency with robots."
}


In [94]:
def score_conversation_facets(
    conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=12
):
    """
    Full evidence-based facet scoring pipeline.

    Pipeline:
    1. Split conversation into statements
    2. Perform hybrid retrieval for each statement
    3. Merge and deduplicate candidate facets
    4. Retrieve complete facet metadata
    5. Apply evidence-based LLM scoring
    6. Validate results
    """

    # ========================================================
    # STEP 1 — STATEMENT-LEVEL RETRIEVAL
    # ========================================================

    candidates = retrieve_facets_by_statement(
        conversation=conversation,
        lexical_k=lexical_k,
        semantic_k=semantic_k,
        max_candidates=max_candidates
    )

    print(
        f"\nRetrieved {len(candidates)} candidate facets."
    )

    print(
        "Starting evidence-based scoring...\n"
    )

    # ========================================================
    # STEP 2 — SCORE EACH CANDIDATE
    # ========================================================

    results = []

    for _, candidate_row in candidates.iterrows():

        facet_id = candidate_row["facet_id"]

        facet_name = candidate_row["raw_facet"]

        print(
            f"Scoring facet {facet_id}: {facet_name}"
        )

        # ====================================================
        # GET COMPLETE FACET METADATA
        #
        # retrieve_facets_by_statement() returns only the
        # retrieval columns. The scorer also needs fields such
        # as scoring_definition and abstention_reason.
        # ====================================================

        full_facet_rows = retrieval_df[
            retrieval_df["facet_id"] == facet_id
        ]

        if full_facet_rows.empty:

            print(
                f"Warning: Facet {facet_id} "
                "was not found in retrieval_df."
            )

            continue

        full_facet_row = (
            full_facet_rows.iloc[0].copy()
        )

        # ====================================================
        # SCORE USING COMPLETE FACET METADATA
        # ====================================================

        scoring_result = score_facet_with_llm(
            conversation=conversation,
            facet_row=full_facet_row
        )

        # ====================================================
        # COMBINE RETRIEVAL + SCORING RESULTS
        # ====================================================

        result = {
            "facet_id": facet_id,
            "facet": facet_name,
            "facet_type": candidate_row[
                "facet_type"
            ],
            "sensitivity": candidate_row[
                "sensitivity"
            ],
            "retrieval_source": candidate_row[
                "retrieval_source"
            ],
            "lexical_similarity": candidate_row[
                "lexical_similarity"
            ],
            "semantic_similarity": candidate_row[
                "semantic_similarity"
            ],
            "source_statement": candidate_row[
                "source_statement"
            ],
            "statement_index": candidate_row[
                "statement_index"
            ],
            "status": scoring_result[
                "status"
            ],
            "score": scoring_result[
                "score"
            ],
            "confidence": scoring_result[
                "confidence"
            ],
            "evidence": scoring_result[
                "evidence"
            ]
        }

        # ====================================================
        # VALIDATION
        # ====================================================

        if result["status"] == "scored":

            validation_passed = (
                result["score"] is not None
                and isinstance(
                    result["score"],
                    (int, float)
                )
                and 1 <= result["score"] <= 5
            )

        else:

            validation_passed = (
                result["score"] is None
            )

        result[
            "validation_passed"
        ] = validation_passed

        results.append(result)

    # ========================================================
    # STEP 3 — FINAL RESULTS
    # ========================================================

    final_results = pd.DataFrame(results)

    return final_results

In [70]:
test_conversation = """
I enjoy taking risks and trying new things, especially when I feel
that it will help me learn and grow. I constantly try to improve myself.

However, I sometimes get easily irritated and annoyed when people
waste my time or repeatedly make the same mistakes.
"""

pipeline_results = score_conversation_facets(
    conversation=test_conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=8
)

display(pipeline_results)

Retrieved 8 candidate facets.
Starting evidence-based scoring...

Scoring facet 66: Creative risk-taking tendency
Scoring facet 1: Risktaking
Scoring facet 379: Creative resilience
Scoring facet 234: Boredom Susceptibility
Scoring facet 275: Impudence
Scoring facet 232: Significance: Desire to make an impact
Scoring facet 350: Wake-time consistency
Scoring facet 341: Time outdoors/day (h)


,facet_id,facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,status,score,confidence,evidence,validation_passed
0,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.274416,0.442333,scored,3.0,0.8,The person mentions enjoying taking risks and ...,True
1,1,Risktaking,behavioral_or_personality,low,semantic,0.000000,0.365602,scored,4.0,0.8,The individual expresses enjoyment in taking r...,True
2,379,Creative resilience,behavioral_or_personality,low,semantic,0.000000,0.357572,scored,3.0,0.8,The individual expresses a willingness to take...,True
3,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.000000,0.344705,scored,3.0,0.5,The person mentions enjoying taking risks and ...,True
4,275,Impudence,behavioral_or_personality,low,semantic,0.000000,0.332322,scored,3.0,0.8,The person mentions enjoying taking risks and ...,True
5,232,Significance: Desire to make an impact,preference_or_attitude,low,lexical,0.264047,0.233986,validation_error,NaN,0.0,Missing evidence explanation.,False
6,350,Wake-time consistency,behavioral_or_personality,low,lexical,0.248038,0.133934,insufficient_evidence,NaN,1.0,The conversation does not discuss wake-time co...,True
7,341,Time outdoors/day (h),behavioral_or_personality,low,lexical,0.257816,0.106501,insufficient_evidence,NaN,1.0,The conversation does not discuss the specific...,True


In [87]:
def score_facet_with_llm(conversation, facet_row):

    # ========================================================
    # BASIC FACET INFORMATION
    # ========================================================

    facet_name = str(facet_row["raw_facet"])
    facet_id = int(facet_row["facet_id"])

    scoring_definition = str(
        facet_row["scoring_definition"]
    )

    abstention_reason = str(
        facet_row["abstention_reason"]
    )

    # ========================================================
    # STEP 1 — DETERMINISTIC LEXICAL / PHRASE GROUNDING
    # ========================================================

    grounding = lexical_grounding_score(
        conversation,
        facet_name
    )

    strong_grounding = grounding["grounded"]

    # ========================================================
    # STEP 2 — STRICT EVIDENCE VERIFICATION
    # ========================================================

    evidence_system_prompt = """
You are a STRICT evidence verifier.

Your job is to determine whether the conversation contains
DIRECT evidence for the requested facet.

IMPORTANT:

Related traits are NOT direct evidence.

Examples:

Risk-taking is not automatically:
- resilience
- confidence
- impudence
- boredom susceptibility

Self-improvement is not automatically:
- desire to make an impact
- resilience
- ambition

Irritability is not automatically:
- boredom susceptibility
- hostility

You may only return directly_about_facet = true when the
conversation explicitly describes:

1. the requested trait itself, OR
2. a concrete behavior that is a direct behavioral manifestation
   of the requested trait.

Do NOT return true simply because something "aligns with",
"is related to", or "suggests" the requested facet.

If evidence requires an inference from one personality trait
to another, return false.

Return JSON only:

{
    "evidence_found": true or false,
    "speaker_statement": "exact supporting statement or null",
    "directly_about_facet": true or false,
    "direct_behavior": "the exact behavior explicitly described or null",
    "reason": "brief explanation"
}
"""

    evidence_user_prompt = f"""
CONVERSATION:

{conversation}


REQUESTED FACET:

{facet_name}


SCORING DEFINITION:

{scoring_definition}


ABSTENTION GUIDANCE:

{abstention_reason}


LEXICAL GROUNDING INFORMATION:

Strong phrase grounding: {strong_grounding}

Matched tokens:
{grounding["matched_tokens"]}

Matched phrases:
{grounding["matched_phrases"]}


TASK:

Determine whether the conversation provides DIRECT evidence
for the requested facet.

If strong lexical grounding is false, you must be especially
conservative.

Do not convert related concepts into the requested trait.

Return JSON only.
"""

    evidence_response = client.chat.completions.create(
        model=SCORING_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": evidence_system_prompt
            },
            {
                "role": "user",
                "content": evidence_user_prompt
            }
        ]
    )

    # ========================================================
    # STEP 3 — PARSE EVIDENCE RESULT
    # ========================================================

    try:

        evidence_result = json.loads(
            evidence_response.choices[0].message.content
        )

    except json.JSONDecodeError:

        return {
            "facet_id": facet_id,
            "status": "insufficient_evidence",
            "score": None,
            "confidence": 1.0,
            "evidence": (
                "Evidence verification output could not be parsed."
            )
        }

    evidence_found = (
        evidence_result.get("evidence_found") is True
    )

    directly_about_facet = (
        evidence_result.get("directly_about_facet") is True
    )

    speaker_statement = evidence_result.get(
        "speaker_statement"
    )

    # ========================================================
    # STEP 4 — STRICT DETERMINISTIC EVIDENCE GATE
    # ========================================================

    grounding_score = grounding["score"]

    matched_tokens = grounding["matched_tokens"]

    matched_phrases = grounding["matched_phrases"]

    # --------------------------------------------------------
    # RULE 1:
    # Explicit phrase/concept grounding is sufficient.
    # --------------------------------------------------------

    has_direct_phrase_grounding = (
        len(matched_phrases) > 0
    )

    # --------------------------------------------------------
    # RULE 2:
    # Otherwise require strong lexical overlap.
    #
    # At least 50% of meaningful facet tokens must occur
    # in the conversation.
    # --------------------------------------------------------

    has_strong_token_grounding = (
        grounding_score >= 0.50
        and len(matched_tokens) >= 1
    )

    # --------------------------------------------------------
    # FINAL DETERMINISTIC GROUNDING DECISION
    # --------------------------------------------------------

    has_deterministic_grounding = (
        has_direct_phrase_grounding
        or has_strong_token_grounding
    )

    # --------------------------------------------------------
    # ABSTAIN IF THERE IS NO DIRECT GROUNDING
    # --------------------------------------------------------

    if not has_deterministic_grounding:

        reason = evidence_result.get(
            "reason",
            "No direct facet-specific evidence was found."
        )

        return {
            "facet_id": facet_id,
            "status": "insufficient_evidence",
            "score": None,
            "confidence": 1.0,
            "evidence": (
                "No sufficient direct lexical or phrase grounding "
                "was found between the conversation and the "
                "requested facet. "
                f"{reason}"
            )
        }

    # --------------------------------------------------------
    # ABSTAIN IF THE LLM VERIFIER DOES NOT CONFIRM EVIDENCE
    # --------------------------------------------------------

    if (
        not evidence_found
        or not directly_about_facet
        or not isinstance(speaker_statement, str)
        or not speaker_statement.strip()
        or speaker_statement.strip().lower() == "null"
    ):

        return {
            "facet_id": facet_id,
            "status": "insufficient_evidence",
            "score": None,
            "confidence": 1.0,
            "evidence": evidence_result.get(
                "reason",
                "No direct facet-specific evidence was found."
            )
        }

    # ========================================================
    # STEP 5 — SCORE ONLY VERIFIED EVIDENCE
    # ========================================================

    scoring_system_prompt = """
You are an evidence-based ordinal evaluator.

The evidence verification stage has confirmed that the
speaker statement directly supports the requested facet.

Score the facet using ONLY the verified statement.

Rules:

1. Do not infer additional traits.
2. Do not use information outside the verified statement.
3. Score must be an integer from 1 to 5.
4. Do not return decimal scores.
5. Confidence must be between 0 and 1.

Return JSON only:

{
    "score": 1,
    "confidence": 0.0,
    "evidence": "brief explanation grounded in the statement"
}
"""

    scoring_user_prompt = f"""
REQUESTED FACET:

{facet_name}


SCORING DEFINITION:

{scoring_definition}


VERIFIED SPEAKER STATEMENT:

"{speaker_statement}"


Score the requested facet using ONLY the
verified speaker statement.

Return JSON only.
"""

    scoring_response = client.chat.completions.create(
        model=SCORING_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": scoring_system_prompt
            },
            {
                "role": "user",
                "content": scoring_user_prompt
            }
        ]
    )

    # ========================================================
    # STEP 6 — PARSE SCORING RESULT
    # ========================================================

    try:

        score_result = json.loads(
            scoring_response.choices[0].message.content
        )

        score = score_result.get("score")
        confidence = score_result.get("confidence")
        evidence = score_result.get("evidence")

        # ====================================================
        # BASIC OUTPUT VALIDATION
        # ====================================================

        if (
            not isinstance(score, int)
            or score < 1
            or score > 5
        ):

            return {
                "facet_id": facet_id,
                "status": "insufficient_evidence",
                "score": None,
                "confidence": 0.0,
                "evidence": (
                    "The model returned an invalid score."
                )
            }

        if not isinstance(confidence, (int, float)):

            confidence = 0.0

        confidence = float(confidence)

        confidence = max(
            0.0,
            min(1.0, confidence)
        )

        if not isinstance(evidence, str) or not evidence.strip():

            evidence = (
                f'Grounded in the statement: "{speaker_statement}"'
            )

        # ====================================================
        # RETURN VALID SCORE
        # ====================================================

        return {
            "facet_id": facet_id,
            "status": "scored",
            "score": score,
            "confidence": confidence,
            "evidence": evidence
        }

    except (
        json.JSONDecodeError,
        TypeError,
        ValueError
    ):

        return {
            "facet_id": facet_id,
            "status": "insufficient_evidence",
            "score": None,
            "confidence": 0.0,
            "evidence": (
                "Scoring output could not be parsed or validated."
            )
        }

In [72]:
pipeline_results = score_conversation_facets(
    conversation=test_conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=8
)

display(pipeline_results)

Retrieved 8 candidate facets.
Starting evidence-based scoring...

Scoring facet 66: Creative risk-taking tendency
Scoring facet 1: Risktaking
Scoring facet 379: Creative resilience
Scoring facet 234: Boredom Susceptibility
Scoring facet 275: Impudence
Scoring facet 232: Significance: Desire to make an impact
Scoring facet 350: Wake-time consistency
Scoring facet 341: Time outdoors/day (h)


,facet_id,facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,status,score,confidence,evidence,validation_passed
0,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.274416,0.442333,scored,4.0,0.8,The individual expresses a preference for taki...,True
1,1,Risktaking,behavioral_or_personality,low,semantic,0.000000,0.365602,scored,4.0,0.8,The individual expresses a preference for taki...,True
2,379,Creative resilience,behavioral_or_personality,low,semantic,0.000000,0.357572,scored,4.0,0.8,The provided quote demonstrates a strong focus...,True
3,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.000000,0.344705,insufficient_evidence,NaN,1.0,The conversation does not explicitly mention o...,True
4,275,Impudence,behavioral_or_personality,low,semantic,0.000000,0.332322,scored,4.0,0.8,The individual expresses a willingness to take...,True
5,232,Significance: Desire to make an impact,preference_or_attitude,low,lexical,0.264047,0.233986,scored,4.0,0.8,The individual explicitly states their desire ...,True
6,350,Wake-time consistency,behavioral_or_personality,low,lexical,0.248038,0.133934,insufficient_evidence,NaN,1.0,The conversation does not mention the speaker'...,True
7,341,Time outdoors/day (h),behavioral_or_personality,low,lexical,0.257816,0.106501,insufficient_evidence,NaN,1.0,The conversation does not mention the specific...,True


In [73]:
problem_facets = [
    "Creative resilience",
    "Impudence",
    "Significance: Desire to make an impact"
]

retrieval_df[
    retrieval_df["raw_facet"].isin(problem_facets)
][
    [
        "facet_id",
        "raw_facet",
        "scoring_definition",
        "abstention_reason",
        "facet_document"
    ]
]

,facet_id,raw_facet,scoring_definition,abstention_reason,facet_document
231,232,Significance: Desire to make an impact,Use a 1-5 ordinal scale based on explicit stat...,Score only when the conversation provides dire...,Facet: significance: desire to make an impact ...
274,275,Impudence,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...,Facet: impudence | Category: behavioral_or_per...
378,379,Creative resilience,Use a 1-5 ordinal scale based on direct conver...,Score only when the conversation provides dire...,Facet: creative resilience | Category: behavio...


In [74]:
problem_facets = [
    "Creative resilience",
    "Impudence",
    "Significance: Desire to make an impact"
]

for _, row in retrieval_df[
    retrieval_df["raw_facet"].isin(problem_facets)
].iterrows():

    print("\n" + "=" * 100)
    print("FACET ID:", row["facet_id"])
    print("FACET:", row["raw_facet"])

    print("\nSCORING DEFINITION:")
    print(row["scoring_definition"])

    print("\nABSTENTION REASON:")
    print(row["abstention_reason"])

    print("\nFACET DOCUMENT:")
    print(row["facet_document"])


FACET ID: 232
FACET: Significance: Desire to make an impact

SCORING DEFINITION:
Use a 1-5 ordinal scale based on explicit statements or repeated direct evidence. Abstain when preference is unclear.

ABSTENTION REASON:
Score only when the conversation provides direct, specific evidence about the speaker.

FACET DOCUMENT:
Facet: significance: desire to make an impact | Category: preference_or_attitude | Observability: conditional | Scoring guidance: Use a 1-5 ordinal scale based on explicit statements or repeated direct evidence. Abstain when preference is unclear.

FACET ID: 275
FACET: Impudence

SCORING DEFINITION:
Use a 1-5 ordinal scale based on direct conversational evidence. Do not infer stable traits from isolated statements.

ABSTENTION REASON:
Score only when the conversation provides direct, specific evidence about the speaker.

FACET DOCUMENT:
Facet: impudence | Category: behavioral_or_personality | Observability: conditional | Scoring guidance: Use a 1-5 ordinal scale based

In [76]:
pipeline_results = score_conversation_facets(
    conversation=test_conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=8
)

display(pipeline_results)

Retrieved 8 candidate facets.
Starting evidence-based scoring...

Scoring facet 66: Creative risk-taking tendency
Scoring facet 1: Risktaking
Scoring facet 379: Creative resilience
Scoring facet 234: Boredom Susceptibility
Scoring facet 275: Impudence
Scoring facet 232: Significance: Desire to make an impact
Scoring facet 350: Wake-time consistency
Scoring facet 341: Time outdoors/day (h)


,facet_id,facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,status,score,confidence,evidence,validation_passed
0,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.274416,0.442333,scored,4.0,0.8,The speaker expresses enjoyment in taking risk...,True
1,1,Risktaking,behavioral_or_personality,low,semantic,0.000000,0.365602,scored,4.0,0.8,The speaker explicitly mentions enjoying risk-...,True
2,379,Creative resilience,behavioral_or_personality,low,semantic,0.000000,0.357572,scored,4.0,0.8,The speaker expresses a willingness to take ri...,True
3,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.000000,0.344705,scored,3.0,0.8,The speaker mentions being easily irritated an...,True
4,275,Impudence,behavioral_or_personality,low,semantic,0.000000,0.332322,insufficient_evidence,NaN,1.0,The statement discusses taking risks and growt...,True
5,232,Significance: Desire to make an impact,preference_or_attitude,low,lexical,0.264047,0.233986,scored,4.0,1.0,The speaker explicitly mentions their desire t...,True
6,350,Wake-time consistency,behavioral_or_personality,low,lexical,0.248038,0.133934,insufficient_evidence,NaN,1.0,The conversation does not mention or discuss w...,True
7,341,Time outdoors/day (h),behavioral_or_personality,low,lexical,0.257816,0.106501,insufficient_evidence,NaN,1.0,The conversation does not mention the exact fa...,True


In [77]:
# Show the complete evidence explanations without truncation

for _, row in pipeline_results.iterrows():
    print("\n" + "=" * 100)
    print("FACET:", row["facet"])
    print("STATUS:", row["status"])
    print("SCORE:", row["score"])
    print("\nEVIDENCE:")
    print(row["evidence"])


FACET: Creative risk-taking tendency
STATUS: scored
SCORE: 4.0

EVIDENCE:
The speaker expresses enjoyment in taking risks and trying new things for personal growth, indicating a high tendency for creative risk-taking.

FACET: Risktaking
STATUS: scored
SCORE: 4.0

EVIDENCE:
The speaker explicitly mentions enjoying risk-taking and trying new things for personal growth.

FACET: Creative resilience
STATUS: scored
SCORE: 4.0

EVIDENCE:
The speaker expresses a willingness to take risks and try new things for personal growth, indicating creative resilience.

FACET: Boredom Susceptibility
STATUS: scored
SCORE: 3.0

EVIDENCE:
The speaker mentions being easily irritated and annoyed, suggesting a higher susceptibility to boredom.

FACET: Impudence
STATUS: insufficient_evidence
SCORE: nan

EVIDENCE:
The statement discusses taking risks and growth, but does not specifically mention impudence.

FACET: Significance: Desire to make an impact
STATUS: scored
SCORE: 4.0

EVIDENCE:
The speaker explicitly

In [78]:
print(test_conversation)


I enjoy taking risks and trying new things, especially when I feel
that it will help me learn and grow. I constantly try to improve myself.

However, I sometimes get easily irritated and annoyed when people
waste my time or repeatedly make the same mistakes.



In [79]:
# Debug the evidence decision for problematic facets

debug_facets = [
    "Creative resilience",
    "Boredom Susceptibility",
    "Significance: Desire to make an impact"
]

for _, facet_row in retrieval_df[
    retrieval_df["raw_facet"].isin(debug_facets)
].iterrows():

    facet_name = str(facet_row["raw_facet"])
    facet_id = int(facet_row["facet_id"])

    evidence_system_prompt = """
You are a strict evidence verifier.

Find an exact statement from the conversation and determine whether
that statement DIRECTLY and SPECIFICALLY describes the requested facet.

Do not infer one trait from another.
Do not use semantic similarity as evidence.
Do not infer stable traits from isolated statements.

Return JSON only:

{
    "evidence_found": true or false,
    "speaker_statement": "exact statement or null",
    "directly_about_facet": true or false,
    "reason": "brief explanation"
}
"""

    evidence_user_prompt = f"""
CONVERSATION:
{test_conversation}

FACET:
{facet_name}

SCORING DEFINITION:
{facet_row["scoring_definition"]}

ABSTENTION GUIDANCE:
{facet_row["abstention_reason"]}
"""

    response = client.chat.completions.create(
        model=SCORING_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": evidence_system_prompt
            },
            {
                "role": "user",
                "content": evidence_user_prompt
            }
        ]
    )

    print("\n" + "=" * 100)
    print(f"FACET ID: {facet_id}")
    print(f"FACET: {facet_name}")
    print(response.choices[0].message.content)


FACET ID: 232
FACET: Significance: Desire to make an impact
{
    "evidence_found": false,
    "speaker_statement": "null",
    "directly_about_facet": false,
    "reason": "The provided conversation does not mention the speaker's desire to make an impact or discuss their significance in a way that directly relates to the given facet."}

FACET ID: 234
FACET: Boredom Susceptibility
{
    "evidence_found": false,
    "speaker_statement": "null",
    "directly_about_facet": false,
    "reason": "The conversation does not mention the speaker being susceptible to boredom or evaluate their ability to tolerate boredom."
}

FACET ID: 379
FACET: Creative resilience
{
    "evidence_found": true,
    "speaker_statement": "I constantly try to improve myself.",
    "directly_about_facet": true,
    "reason": "The speaker explicitly mentions their desire to improve themselves, which aligns with the facet of creative resilience."
}


In [80]:
facet_name = "Creative resilience"

test_prompt = f"""
You are performing a STRICT evidence equivalence check.

CONVERSATION:
{test_conversation}

REQUESTED FACET:
{facet_name}

TASK:

Step 1:
Identify the exact behavior, preference, attitude, ability, or fact
explicitly stated in the conversation.

Do not infer any additional trait.

Step 2:
Compare that explicit behavior with the requested facet.

The requested facet can be considered directly supported ONLY if the
conversation explicitly describes the same behavior or trait.

Do NOT use:
- semantic similarity
- related personality traits
- inferred characteristics
- causal assumptions
- interpretations beyond what is explicitly stated

Return JSON only:

{{
    "explicit_behavior": "what is literally described",
    "requested_facet": "{facet_name}",
    "same_trait": true or false,
    "reason": "brief explanation"
}}
"""

response = client.chat.completions.create(
    model=SCORING_MODEL,
    temperature=0,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a strict evidence equivalence verifier. Do not infer traits."
        },
        {
            "role": "user",
            "content": test_prompt
        }
    ]
)

print(response.choices[0].message.content)

{
    "explicit_behavior": "enjoys taking risks, trying new things, improving oneself",
    "requested_facet": "Creative resilience",
    "same_trait": true,
    "reason": "The conversation explicitly describes the individual's willingness to take risks, learn, and improve, which aligns with the facet of creative resilience."}


In [81]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


def normalize_text(text):
    """
    Normalize text for simple lexical grounding checks.
    """
    text = str(text).lower()

    # Normalize common hyphen variations
    text = text.replace("-", " ")

    # Keep alphabetic characters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Collapse repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_meaningful(text):
    """
    Extract meaningful tokens while removing common stop words.
    """
    text = normalize_text(text)

    tokens = text.split()

    return {
        token
        for token in tokens
        if token not in ENGLISH_STOP_WORDS
        and len(token) > 2
    }


def lexical_grounding_score(conversation, facet_name):
    """
    Measure direct lexical overlap between the conversation
    and the requested facet.
    """

    conversation_tokens = tokenize_meaningful(conversation)
    facet_tokens = tokenize_meaningful(facet_name)

    if not facet_tokens:
        return 0.0, []

    matched_tokens = conversation_tokens.intersection(facet_tokens)

    score = len(matched_tokens) / len(facet_tokens)

    return score, sorted(matched_tokens)

In [82]:
test_facets = [
    "Risktaking",
    "Creative risk-taking tendency",
    "Creative resilience",
    "Boredom Susceptibility",
    "Impudence",
    "Significance: Desire to make an impact"
]

for facet in test_facets:

    score, matches = lexical_grounding_score(
        test_conversation,
        facet
    )

    print(f"\nFacet: {facet}")
    print("Grounding score:", score)
    print("Matched tokens:", matches)


Facet: Risktaking
Grounding score: 0.0
Matched tokens: []

Facet: Creative risk-taking tendency
Grounding score: 0.25
Matched tokens: ['taking']

Facet: Creative resilience
Grounding score: 0.0
Matched tokens: []

Facet: Boredom Susceptibility
Grounding score: 0.0
Matched tokens: []

Facet: Impudence
Grounding score: 0.0
Matched tokens: []

Facet: Significance: Desire to make an impact
Grounding score: 0.25
Matched tokens: ['make']


In [83]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

def normalize_text(text):

    text = str(text).lower()

    # Normalize common hyphen forms
    text = text.replace("-", " ")

    # Keep letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ------------------------------------------------------------
# Simple word normalization
# ------------------------------------------------------------

def normalize_token(token):

    token = token.lower()

    # Basic plural normalization
    if token.endswith("ies") and len(token) > 4:
        token = token[:-3] + "y"

    elif token.endswith("s") and not token.endswith("ss") and len(token) > 3:
        token = token[:-1]

    return token


def tokenize_meaningful(text):

    text = normalize_text(text)

    tokens = text.split()

    normalized_tokens = set()

    for token in tokens:

        token = normalize_token(token)

        if (
            token not in ENGLISH_STOP_WORDS
            and len(token) > 2
        ):
            normalized_tokens.add(token)

    return normalized_tokens


# ------------------------------------------------------------
# Compound concept handling
# ------------------------------------------------------------

CONCEPT_VARIANTS = {

    "risktaking": [
        "risk taking",
        "taking risks",
        "take risks",
        "takes risks"
    ],

    "risk taking": [
        "taking risks",
        "take risks",
        "takes risks"
    ]
}


def normalize_facet_name(facet_name):

    facet = normalize_text(facet_name)

    # Remove spaces for compound matching
    compact = facet.replace(" ", "")

    if compact == "risktaking":
        return "risktaking"

    return facet


# ------------------------------------------------------------
# Phrase grounding
# ------------------------------------------------------------

def phrase_grounding_matches(conversation, facet_name):

    conversation_normalized = normalize_text(conversation)

    facet_normalized = normalize_facet_name(facet_name)

    matches = []

    # Direct phrase match
    if facet_normalized in conversation_normalized:
        matches.append(facet_normalized)

    # Known compound variants
    if facet_normalized in CONCEPT_VARIANTS:

        for variant in CONCEPT_VARIANTS[facet_normalized]:

            if variant in conversation_normalized:
                matches.append(variant)

    return matches


# ------------------------------------------------------------
# Final lexical grounding score
# ------------------------------------------------------------

def lexical_grounding_score(conversation, facet_name):

    conversation_tokens = tokenize_meaningful(conversation)

    facet_tokens = tokenize_meaningful(facet_name)

    matched_tokens = (
        conversation_tokens.intersection(facet_tokens)
    )

    phrase_matches = phrase_grounding_matches(
        conversation,
        facet_name
    )

    # Phrase match means strong direct grounding
    if phrase_matches:

        return {
            "grounded": True,
            "score": 1.0,
            "matched_tokens": sorted(matched_tokens),
            "matched_phrases": phrase_matches
        }

    # Token overlap ratio
    if not facet_tokens:

        return {
            "grounded": False,
            "score": 0.0,
            "matched_tokens": [],
            "matched_phrases": []
        }

    score = len(matched_tokens) / len(facet_tokens)

    return {
        "grounded": False,
        "score": score,
        "matched_tokens": sorted(matched_tokens),
        "matched_phrases": []
    }

In [84]:
test_facets = [
    "Risktaking",
    "Creative risk-taking tendency",
    "Creative resilience",
    "Boredom Susceptibility",
    "Impudence",
    "Significance: Desire to make an impact"
]

for facet in test_facets:

    result = lexical_grounding_score(
        test_conversation,
        facet
    )

    print("\nFacet:", facet)
    print("Grounded:", result["grounded"])
    print("Score:", result["score"])
    print("Matched tokens:", result["matched_tokens"])
    print("Matched phrases:", result["matched_phrases"])


Facet: Risktaking
Grounded: True
Score: 1.0
Matched tokens: []
Matched phrases: ['taking risks']

Facet: Creative risk-taking tendency
Grounded: False
Score: 0.5
Matched tokens: ['risk', 'taking']
Matched phrases: []

Facet: Creative resilience
Grounded: False
Score: 0.0
Matched tokens: []
Matched phrases: []

Facet: Boredom Susceptibility
Grounded: False
Score: 0.0
Matched tokens: []
Matched phrases: []

Facet: Impudence
Grounded: False
Score: 0.0
Matched tokens: []
Matched phrases: []

Facet: Significance: Desire to make an impact
Grounded: False
Score: 0.25
Matched tokens: ['make']
Matched phrases: []


In [88]:
pipeline_results = score_conversation_facets(
    conversation=test_conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=8
)

display(pipeline_results)

Retrieved 8 candidate facets.
Starting evidence-based scoring...

Scoring facet 66: Creative risk-taking tendency
Scoring facet 1: Risktaking
Scoring facet 379: Creative resilience
Scoring facet 234: Boredom Susceptibility
Scoring facet 275: Impudence
Scoring facet 232: Significance: Desire to make an impact
Scoring facet 350: Wake-time consistency
Scoring facet 341: Time outdoors/day (h)


,facet_id,facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,status,score,confidence,evidence,validation_passed
0,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.274416,0.442333,scored,4.0,0.9,The speaker expresses enjoyment in taking risk...,True
1,1,Risktaking,behavioral_or_personality,low,semantic,0.000000,0.365602,scored,4.0,0.9,The speaker explicitly mentions enjoying takin...,True
2,379,Creative resilience,behavioral_or_personality,low,semantic,0.000000,0.357572,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
3,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.000000,0.344705,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
4,275,Impudence,behavioral_or_personality,low,semantic,0.000000,0.332322,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
5,232,Significance: Desire to make an impact,preference_or_attitude,low,lexical,0.264047,0.233986,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
6,350,Wake-time consistency,behavioral_or_personality,low,lexical,0.248038,0.133934,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
7,341,Time outdoors/day (h),behavioral_or_personality,low,lexical,0.257816,0.106501,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True


In [89]:
import inspect

print(inspect.getsource(retrieve_facets_hybrid))

def retrieve_facets_hybrid(query, lexical_k=5, semantic_k=5):
    """
    Retrieve candidate facets using both lexical TF-IDF and
    semantic embedding similarity.

    The two retrieval methods are combined using candidate union
    rather than directly averaging their scores.
    """

    # ----------------------------
    # 1. Lexical retrieval
    # ----------------------------
    query_vector = vectorizer.transform([query])

    lexical_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    lexical_indices = np.argsort(
        lexical_scores
    )[::-1][:lexical_k]

    # Keep only positive lexical matches
    lexical_indices = [
        int(idx)
        for idx in lexical_indices
        if lexical_scores[idx] > 0
    ]

    # ----------------------------
    # 2. Semantic retrieval
    # ----------------------------
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=Tr

In [90]:
import re
import pandas as pd


def split_conversation_into_statements(conversation):
    """
    Split a conversation into meaningful statements.

    This is intentionally simple and deterministic.
    """

    # Normalize whitespace
    conversation = re.sub(
        r"\s+",
        " ",
        str(conversation)
    ).strip()

    # Split on sentence-ending punctuation
    statements = re.split(
        r"(?<=[.!?])\s+",
        conversation
    )

    # Keep meaningful statements
    statements = [
        statement.strip()
        for statement in statements
        if len(statement.strip()) > 10
    ]

    return statements


def retrieve_facets_by_statement(
    conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=12
):
    """
    Retrieve facets separately for each statement in the conversation.

    Candidates from all statements are combined, deduplicated,
    and ranked using their best retrieval scores.
    """

    statements = split_conversation_into_statements(
        conversation
    )

    all_results = []

    print(
        f"Split conversation into {len(statements)} statement(s)."
    )

    # ========================================================
    # RETRIEVE FOR EACH STATEMENT
    # ========================================================

    for statement_index, statement in enumerate(
        statements,
        start=1
    ):

        print(
            f"\nRetrieving for statement {statement_index}:"
        )
        print(f'"{statement}"')

        statement_results = retrieve_facets_hybrid(
            statement,
            lexical_k=lexical_k,
            semantic_k=semantic_k
        ).copy()

        # Keep track of which statement retrieved each facet
        statement_results[
            "source_statement"
        ] = statement

        statement_results[
            "statement_index"
        ] = statement_index

        all_results.append(
            statement_results
        )

    # ========================================================
    # COMBINE RESULTS
    # ========================================================

    if not all_results:

        return pd.DataFrame()

    combined_results = pd.concat(
        all_results,
        ignore_index=True
    )

    # ========================================================
    # DEDUPLICATE FACETS
    #
    # A facet can be retrieved by multiple statements.
    # Keep the occurrence with the highest semantic similarity.
    # ========================================================

    combined_results = (
        combined_results
        .sort_values(
            by=[
                "semantic_similarity",
                "lexical_similarity"
            ],
            ascending=False
        )
        .drop_duplicates(
            subset="facet_id",
            keep="first"
        )
    )

    # ========================================================
    # FINAL RANKING
    #
    # We use the best semantic similarity as the main ranking
    # signal and lexical similarity as a secondary signal.
    # ========================================================

    combined_results = (
        combined_results
        .sort_values(
            by=[
                "semantic_similarity",
                "lexical_similarity"
            ],
            ascending=False
        )
        .head(max_candidates)
        .reset_index(drop=True)
    )

    return combined_results

In [91]:
statement_candidates = retrieve_facets_by_statement(
    test_conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=12
)

display(statement_candidates)

Split conversation into 3 statement(s).

Retrieving for statement 1:
"I enjoy taking risks and trying new things, especially when I feel that it will help me learn and grow."

Retrieving for statement 2:
"I constantly try to improve myself."

Retrieving for statement 3:
"However, I sometimes get easily irritated and annoyed when people waste my time or repeatedly make the same mistakes."


,facet_id,raw_facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,source_statement,statement_index
0,11,Self-improvement,behavioral_or_personality,low,semantic,0.00000,0.459313,I constantly try to improve myself.,2
1,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.37762,0.445088,"I enjoy taking risks and trying new things, es...",1
2,361,Irritability,behavioral_or_personality,low,semantic,0.00000,0.401576,"However, I sometimes get easily irritated and ...",3
3,1,Risktaking,behavioral_or_personality,low,semantic,0.00000,0.399105,"I enjoy taking risks and trying new things, es...",1
4,301,Psychological construct: Excuse-Making tendency,behavioral_or_personality,low,semantic,0.00000,0.352971,"However, I sometimes get easily irritated and ...",3
5,379,Creative resilience,behavioral_or_personality,low,semantic,0.00000,0.342754,"I enjoy taking risks and trying new things, es...",1
6,389,Patience: Resistance to anger,behavioral_or_personality,low,semantic,0.00000,0.338589,"However, I sometimes get easily irritated and ...",3
7,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.00000,0.332683,"However, I sometimes get easily irritated and ...",3
8,304,Social-cognition variable: Faux Pas Recognitio...,behavioral_or_personality,low,semantic,0.00000,0.322757,"However, I sometimes get easily irritated and ...",3
9,15,Adventure-Seeking Behavior,behavioral_or_personality,low,semantic,0.00000,0.313286,"I enjoy taking risks and trying new things, es...",1


In [95]:
pipeline_results = score_conversation_facets(
    conversation=test_conversation,
    lexical_k=5,
    semantic_k=5,
    max_candidates=12
)

display(pipeline_results)

Split conversation into 3 statement(s).

Retrieving for statement 1:
"I enjoy taking risks and trying new things, especially when I feel that it will help me learn and grow."

Retrieving for statement 2:
"I constantly try to improve myself."

Retrieving for statement 3:
"However, I sometimes get easily irritated and annoyed when people waste my time or repeatedly make the same mistakes."

Retrieved 12 candidate facets.
Starting evidence-based scoring...

Scoring facet 11: Self-improvement
Scoring facet 66: Creative risk-taking tendency
Scoring facet 361: Irritability
Scoring facet 1: Risktaking
Scoring facet 301: Psychological construct: Excuse-Making tendency
Scoring facet 379: Creative resilience
Scoring facet 389: Patience: Resistance to anger
Scoring facet 234: Boredom Susceptibility
Scoring facet 304: Social-cognition variable: Faux Pas Recognition accuracy
Scoring facet 15: Adventure-Seeking Behavior
Scoring facet 267: Fearfulness: Fear of physical dangers
Scoring facet 391: Self

,facet_id,facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,source_statement,statement_index,status,score,confidence,evidence,validation_passed
0,11,Self-improvement,behavioral_or_personality,low,semantic,0.00000,0.459313,I constantly try to improve myself.,2,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
1,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.37762,0.445088,"I enjoy taking risks and trying new things, es...",1,scored,4.0,0.9,The speaker expresses enjoyment in taking risk...,True
2,361,Irritability,behavioral_or_personality,low,semantic,0.00000,0.401576,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
3,1,Risktaking,behavioral_or_personality,low,semantic,0.00000,0.399105,"I enjoy taking risks and trying new things, es...",1,scored,4.0,0.9,The speaker explicitly mentions enjoying takin...,True
4,301,Psychological construct: Excuse-Making tendency,behavioral_or_personality,low,semantic,0.00000,0.352971,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
5,379,Creative resilience,behavioral_or_personality,low,semantic,0.00000,0.342754,"I enjoy taking risks and trying new things, es...",1,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
6,389,Patience: Resistance to anger,behavioral_or_personality,low,semantic,0.00000,0.338589,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
7,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.00000,0.332683,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
8,304,Social-cognition variable: Faux Pas Recognitio...,behavioral_or_personality,low,semantic,0.00000,0.322757,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
9,15,Adventure-Seeking Behavior,behavioral_or_personality,low,semantic,0.00000,0.313286,"I enjoy taking risks and trying new things, es...",1,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True


In [98]:
import re
from nltk.stem import PorterStemmer


# ============================================================
# INITIALIZE STEMMER
# ============================================================

stemmer = PorterStemmer()


# ============================================================
# DIRECT PHRASE ALIASES
#
# These aliases represent direct linguistic variations of the
# same facet. They are NOT semantic relationships between
# different personality traits.
# ============================================================

FACET_PHRASE_ALIASES = {
    "risktaking": [
        "taking risks",
        "take risks",
        "risk taking"
    ],

    "self-improvement": [
        "improve myself",
        "improving myself",
        "self improvement"
    ],

    "irritability": [
        "easily irritated",
        "get irritated",
        "feeling irritated",
        "get easily irritated"
    ]
}


# ============================================================
# LEXICAL GROUNDING FUNCTION
# ============================================================

def lexical_grounding_score(
    conversation,
    facet_name
):
    """
    Determine whether a facet has direct lexical grounding
    in a conversation.

    Grounding methods:
    1. Exact facet phrase matching
    2. Explicit phrase alias matching
    3. Stemmed token overlap

    The function intentionally does NOT treat semantic
    similarity between different traits as direct evidence.
    """

    # ========================================================
    # NORMALIZE TEXT
    # ========================================================

    conversation_lower = (
        str(conversation)
        .lower()
        .strip()
    )

    facet_lower = (
        str(facet_name)
        .lower()
        .strip()
    )


    # ========================================================
    # TOKENIZATION
    # ========================================================

    conversation_tokens = re.findall(
        r"[a-zA-Z]+",
        conversation_lower
    )

    facet_tokens = re.findall(
        r"[a-zA-Z]+",
        facet_lower
    )


    # ========================================================
    # STEM TOKENS
    # ========================================================

    conversation_stems = {
        stemmer.stem(token)
        for token in conversation_tokens
    }

    facet_stems = {
        stemmer.stem(token)
        for token in facet_tokens
    }


    # ========================================================
    # EXACT FACET PHRASE MATCHING
    # ========================================================

    matched_phrases = []

    if (
        facet_lower
        and facet_lower in conversation_lower
    ):
        matched_phrases.append(
            facet_lower
        )


    # ========================================================
    # DIRECT PHRASE ALIAS MATCHING
    # ========================================================

    facet_aliases = FACET_PHRASE_ALIASES.get(
        facet_lower,
        []
    )

    for alias in facet_aliases:

        if alias in conversation_lower:

            matched_phrases.append(
                alias
            )


    # ========================================================
    # STEMMED TOKEN MATCHING
    # ========================================================

    matched_stems = (
        conversation_stems
        .intersection(facet_stems)
    )

    matched_tokens = [
        token
        for token in facet_tokens
        if stemmer.stem(token) in matched_stems
    ]


    # ========================================================
    # GROUNDING SCORE
    # ========================================================

    if len(facet_stems) > 0:

        grounding_score = (
            len(matched_stems)
            / len(facet_stems)
        )

    else:

        grounding_score = 0.0


    # ========================================================
    # FINAL GROUNDING DECISION
    # ========================================================

    grounded = (
        len(matched_phrases) > 0
        or grounding_score >= 0.50
    )


    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {
        "grounded": grounded,
        "score": grounding_score,
        "matched_tokens": matched_tokens,
        "matched_phrases": matched_phrases
    }

In [100]:
test_facets = [
    "Self-improvement",
    "Irritability",
    "Risktaking",
    "Creative resilience"
]

for facet in test_facets:

    result = lexical_grounding_score(
        test_conversation,
        facet
    )

    print("\nFacet:", facet)
    print("Grounded:", result["grounded"])
    print("Score:", result["score"])
    print("Matched tokens:", result["matched_tokens"])
    print("Matched phrases:", result["matched_phrases"])


Facet: Self-improvement
Grounded: True
Score: 0.5
Matched tokens: ['improvement']
Matched phrases: ['improve myself']

Facet: Irritability
Grounded: True
Score: 1.0
Matched tokens: ['irritability']
Matched phrases: ['easily irritated', 'get easily irritated']

Facet: Risktaking
Grounded: True
Score: 0.0
Matched tokens: []
Matched phrases: ['taking risks']

Facet: Creative resilience
Grounded: False
Score: 0.0
Matched tokens: []
Matched phrases: []


In [101]:
conversation = """
I enjoy taking risks and trying new things, especially when I feel that it will help me learn and grow.
I constantly try to improve myself.
However, I sometimes get easily irritated and annoyed when people waste my time or repeatedly make the same mistakes.
"""


In [102]:
results = score_conversation_facets(
    conversation,
    lexical_k=5,
    semantic_k=5
)

display(results)

Split conversation into 3 statement(s).

Retrieving for statement 1:
"I enjoy taking risks and trying new things, especially when I feel that it will help me learn and grow."

Retrieving for statement 2:
"I constantly try to improve myself."

Retrieving for statement 3:
"However, I sometimes get easily irritated and annoyed when people waste my time or repeatedly make the same mistakes."

Retrieved 12 candidate facets.
Starting evidence-based scoring...

Scoring facet 11: Self-improvement
Scoring facet 66: Creative risk-taking tendency
Scoring facet 361: Irritability
Scoring facet 1: Risktaking
Scoring facet 301: Psychological construct: Excuse-Making tendency
Scoring facet 379: Creative resilience
Scoring facet 389: Patience: Resistance to anger
Scoring facet 234: Boredom Susceptibility
Scoring facet 304: Social-cognition variable: Faux Pas Recognition accuracy
Scoring facet 15: Adventure-Seeking Behavior
Scoring facet 267: Fearfulness: Fear of physical dangers
Scoring facet 391: Self

,facet_id,facet,facet_type,sensitivity,retrieval_source,lexical_similarity,semantic_similarity,source_statement,statement_index,status,score,confidence,evidence,validation_passed
0,11,Self-improvement,behavioral_or_personality,low,semantic,0.00000,0.459313,I constantly try to improve myself.,2,scored,5.0,1.0,The speaker explicitly mentions their dedicati...,True
1,66,Creative risk-taking tendency,behavioral_or_personality,low,hybrid,0.37762,0.445088,"I enjoy taking risks and trying new things, es...",1,scored,4.0,0.9,The speaker expresses enjoyment in taking risk...,True
2,361,Irritability,behavioral_or_personality,low,semantic,0.00000,0.401576,"However, I sometimes get easily irritated and ...",3,scored,4.0,0.9,The statement directly indicates the speaker e...,True
3,1,Risktaking,behavioral_or_personality,low,semantic,0.00000,0.399105,"I enjoy taking risks and trying new things, es...",1,scored,4.0,0.9,The speaker explicitly mentions enjoying takin...,True
4,301,Psychological construct: Excuse-Making tendency,behavioral_or_personality,low,semantic,0.00000,0.352971,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
5,379,Creative resilience,behavioral_or_personality,low,semantic,0.00000,0.342754,"I enjoy taking risks and trying new things, es...",1,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
6,389,Patience: Resistance to anger,behavioral_or_personality,low,semantic,0.00000,0.338589,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
7,234,Boredom Susceptibility,behavioral_or_personality,low,semantic,0.00000,0.332683,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
8,304,Social-cognition variable: Faux Pas Recognitio...,behavioral_or_personality,low,semantic,0.00000,0.322757,"However, I sometimes get easily irritated and ...",3,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True
9,15,Adventure-Seeking Behavior,behavioral_or_personality,low,semantic,0.00000,0.313286,"I enjoy taking risks and trying new things, es...",1,insufficient_evidence,NaN,1.0,No sufficient direct lexical or phrase groundi...,True


In [103]:
# ============================================================
# FINAL SCORED FACETS SUMMARY
# ============================================================

scored_facets = results[
    results["status"] == "scored"
].copy()

scored_facets = scored_facets[
    [
        "facet_id",
        "facet",
        "score",
        "confidence",
        "evidence",
        "source_statement"
    ]
]

display(scored_facets)

,facet_id,facet,score,confidence,evidence,source_statement
0,11,Self-improvement,5.0,1.0,The speaker explicitly mentions their dedicati...,I constantly try to improve myself.
1,66,Creative risk-taking tendency,4.0,0.9,The speaker expresses enjoyment in taking risk...,"I enjoy taking risks and trying new things, es..."
2,361,Irritability,4.0,0.9,The statement directly indicates the speaker e...,"However, I sometimes get easily irritated and ..."
3,1,Risktaking,4.0,0.9,The speaker explicitly mentions enjoying takin...,"I enjoy taking risks and trying new things, es..."


# Phase 3: Evidence-Based Facet Scoring

This phase converts retrieved candidate facets into structured
facet scores using an evidence-based approach.

A candidate retrieved through semantic or lexical similarity is not
automatically treated as evidence about the speaker.

The pipeline therefore applies:

1. Direct lexical and phrase grounding.
2. Evidence verification against the source statement.
3. LLM-based scoring only when sufficient direct evidence exists.
4. Structured JSON output validation.
5. Abstention when evidence is insufficient.

This reduces false positive scoring caused by semantically related but
unsupported candidate facets.